<a href="https://colab.research.google.com/github/wonsang/CalculusQuiz/blob/master/Copy_of_3%EC%A1%B0_WordPiece_and_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔤 토큰화 기초 — Part 2: WordPiece 어휘 집합 구축 & BPE vs WordPiece 비교 (Lecture 03)
**딥러닝응용III** | 동덕여자대학교 데이터사이언스전공 | 유원상 교수

> **Part 1**에서 Byte-Level BPE 토크나이저를 직접 구축했습니다.  
> **Part 2**에서는 BERT가 사용하는 WordPiece 토크나이저를 구축하고,  
> BPE와 WordPiece가 실제로 어떻게 다르게 동작하는지 직접 비교합니다.

---

## 🎯 강의 목표

1. **WordPiece 어휘 집합 구축** — `BertWordPieceTokenizer`를 직접 학습하고 `vocab.txt`를 이해한다
2. **BERT 입력값 생성** — `input_ids`, `attention_mask`, `token_type_ids`의 의미와 생성 방법을 이해한다
3. **BPE vs WordPiece 비교** — 두 알고리즘의 설계 철학과 실제 토큰화 결과 차이를 파악한다

---

## 🗺️ 노트북 구성

| 단계 | 내용 | 핵심 개념 |
|------|------|----------|
| 1단계 | 실습 환경 만들기 | 패키지 설치, 구글 드라이브 연동, 말뭉치 준비 |
| 2단계 | WordPiece 어휘 집합 구축 | BertWordPieceTokenizer, vocab.txt, `##` prefix |
| 3단계 | BERT 입력값 만들기 | `[CLS]`/`[SEP]`, input_ids, attention_mask, token_type_ids |
| 4단계 | BPE vs WordPiece 비교 | 병합 기준, 토큰 표현 방식, 정량 비교 |
| 🔧 Remix | 변형 실험 구역 | 학생 작성 |
| 🎬 Demo | 미니 데모 구역 | 학생 작성 |

---

> 💡 **Part 1 노트북과의 연속성**  
> Part 1에서 학습·저장한 `BBPE_DIR`(`/gdrive/My Drive/nlpbook/bbpe`)의 파일들을  
> 이 노트북의 4단계 비교 실습에서 그대로 사용합니다.

---

> ⚠️ **조별 실습 리드 팀 안내**  
> Part 2의 Remix·Demo 주제는 **Part 1(BPE)을 담당한 팀과 겹치지 않도록** 설계되었습니다.  
> Part 2 리드 팀은 WordPiece 고유의 특성(`##` prefix, `[MASK]`, NSP, `[UNK]` 처리 등)에  
> 집중하여 실험을 구성하세요.


---
## 📋 조별 실습 리드 가이드

> **운영 방법**: 4개 팀이 3회에 걸쳐 실습을 리드합니다.  
> 각 조는 **① Explain → ② Remix → ③ Demo** 순서로 진행하세요.  
> 평가: 교수 평가 60% + 학생 상호평가 40%

---

### ① Explain (설명, 20점) — 핵심 셀 설명

강의 내용과 연계하여 아래 핵심 셀들을 설명하세요.  
*(발표 슬라이드 + 실습 Colab을 사전 공유)*

| 설명 항목 | 설명 포인트 |
|----------|------------|
| WordPiece 학습 셀 | `BertWordPieceTokenizer(lowercase=…)`와 `special_tokens` 5종의 의미 |
| vocab.txt 산출물 | 줄 번호 = 인덱스, `##` prefix 토큰이 만들어지는 원리 |
| BERT 토크나이저 선언 | `do_lower_case=False`, 학습 설정과 반드시 일치시켜야 하는 이유 |
| BERT input_ids | `[CLS]`/`[SEP]`가 자동 추가되는 이유와 역할 |
| token_type_ids | 단일 문장(모두 0) vs 두 문장(0/1 분리) 비교 |
| BPE vs WordPiece | 병합 기준, 저장 방식, 파일 구조, OOV 처리 차이 |

---

### ② Remix (변형, 50점) — WordPiece 고유 실험

> Part 1(BPE) 리드 팀과 **겹치지 않는** WordPiece 고유 실험 주제입니다.

| 주제 | BPE 팀과의 차별점 |
|------|-----------------|
| A. `##` prefix 비율 심층 분석 | BPE에는 `##` 개념 자체가 없음 |
| B. `lowercase` 옵션 비교 | BPE에 없는 WordPiece 고유 파라미터 |
| C. 두 문장 입력 & NSP 탐구 | BERT 고유 기능, GPT에는 없음 |
| D. `[MASK]` 토큰 삽입 실험 | MLM 학습 시뮬레이션, BPE에 없음 |
| E. `[UNK]` 발생 패턴 분석 | WordPiece의 OOV 처리 특성 |

→ 노트북 하단 **🔧 Remix 실험 구역** 셀을 활용하세요.

---

### ③ Demo (데모, 20점) — BPE ↔ WordPiece 나란히 비교

> BPE 리드 팀의 Demo가 "BPE 단독 시각화"였다면,  
> WordPiece 리드 팀의 Demo는 **두 토크나이저를 동시에 비교**하는 데 초점을 맞춥니다.

- **입력**: 문장 1개
- **출력**: BPE 결과(▁ 어절 경계 + 한국어 복원)와 WordPiece 결과(`##` prefix 강조)를 나란히 표시
- **핵심 메시지**: 같은 어절이 두 방식에서 어떻게 다르게 분리되는지를 시각적으로 보여주세요

→ 노트북 하단 **🎬 Demo 구역** 셀을 활용하세요.

---

### 📊 평가 기준

| 항목 | 배점 | 기준 |
|------|------|------|
| Explain | 20 | 핵심을 정확히 짚었는가 |
| Remix | 50 | 코드 수정의 실질성 + 실험 설계/결과 해석 |
| Demo | 20 | 실제로 작동하고 사용 시나리오가 명확한가 |
| 재현성/정리 | 10 | 실행 방법, 설정값, 결과 캡처 |

---


---
## 1단계: 실습 환경 만들기

### ⚙️ 런타임 설정 (먼저 확인!)

상단 메뉴 → **런타임** → **런타임 유형 변경** → 하드웨어 가속기 → **None (CPU)**

---

### 📦 필요한 패키지

| 패키지 | Part 2에서의 역할 |
|--------|-----------------|
| `korpora` | NSMC 말뭉치 재로딩 (런타임 재시작 시) |
| `tokenizers` | `BertWordPieceTokenizer`로 WordPiece 어휘 집합 학습 |
| `transformers` | `BertTokenizer`, `GPT2Tokenizer`로 모델 입력값 생성 및 비교 |


In [ ]:
!pip install -q korpora tokenizers transformers

import tokenizers, transformers
print(f"tokenizers   버전: {tokenizers.__version__}")
print(f"transformers 버전: {transformers.__version__}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.8 MB/s eta 0:00:00
tokenizers   버전: 0.22.2
transformers 버전: 5.0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---

### 🔗 구글 드라이브 연동 및 경로 설정

Part 1에서 저장한 BBPE 결과와 이번 실습에서 저장할 WordPiece 결과를 모두 구글 드라이브에서 관리합니다.


In [ ]:
from google.colab import drive
import os

drive.mount('/gdrive', force_remount=True)

BBPE_DIR      = "/gdrive/My Drive/nlpbook/bbpe"       # Part 1 결과
WORDPIECE_DIR = "/gdrive/My Drive/nlpbook/wordpiece"  # Part 2 저장 위치

os.makedirs(WORDPIECE_DIR, exist_ok=True)

print(f"✅ 구글 드라이브 연결 완료")
print(f"   BBPE 경로      (Part 1): {BBPE_DIR}")
print(f"   WordPiece 경로 (Part 2): {WORDPIECE_DIR}")

bbpe_files = os.listdir(BBPE_DIR) if os.path.exists(BBPE_DIR) else []
if bbpe_files:
    print(f"\n✅ Part 1 BBPE 파일 확인: {bbpe_files}")
else:
    print("\n⚠️  Part 1 BBPE 파일이 없습니다. 4단계 비교 전에 Part 1을 먼저 실행하세요.")


Mounted at /gdrive
✅ 구글 드라이브 연결 완료
   BBPE 경로      (Part 1): /gdrive/My Drive/nlpbook/bbpe
   WordPiece 경로 (Part 2): /gdrive/My Drive/nlpbook/wordpiece

✅ Part 1 BBPE 파일 확인: ['vocab.json', 'merges.txt', '현재_나의조_최종']


---

### 📂 말뭉치 준비

BPE와 WordPiece를 **동일한 조건**에서 비교하기 위해 같은 NSMC 말뭉치를 사용합니다.  
런타임이 재시작됐다면 자동으로 재다운로드합니다.


In [ ]:
TRAIN_PATH = "/root/train.txt"
TEST_PATH  = "/root/test.txt"

def write_lines(path, lines):
    with open(path, 'w', encoding='utf-8') as f:
        for line in lines:
            f.write(f'{line}\n')

if not os.path.exists(TRAIN_PATH) or not os.path.exists(TEST_PATH):
    print("말뭉치 파일이 없습니다. NSMC를 다운로드합니다...")
    from Korpora import Korpora
    nsmc = Korpora.load("nsmc", force_download=False)
    write_lines(TRAIN_PATH, nsmc.train.get_all_texts())
    write_lines(TEST_PATH,  nsmc.test.get_all_texts())
    print("✅ 다운로드 및 저장 완료")
else:
    print("✅ 기존 말뭉치 파일을 재사용합니다.")

train_mb = os.path.getsize(TRAIN_PATH) / (1024*1024)
test_mb  = os.path.getsize(TEST_PATH)  / (1024*1024)
print(f"   train.txt : {train_mb:.1f} MB  /  test.txt : {test_mb:.1f} MB")


말뭉치 파일이 없습니다. NSMC를 다운로드합니다...

    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 176MB/s]
[nsmc] download ratings_test.txt: 4.90MB [00:00, 94.7MB/s]


✅ 다운로드 및 저장 완료
   train.txt : 12.5 MB  /  test.txt : 4.2 MB


---
## 2단계: WordPiece 어휘 집합 구축

### 🔬 WordPiece란?

WordPiece는 BPE와 동일하게 서브워드 단위 토큰화를 수행하지만, **병합 기준이 다릅니다.**

| 항목 | BPE | WordPiece |
|------|-----|-----------|
| 병합 기준 | **빈도** 최대화: `count(AB)`가 가장 큰 쌍 선택 | **우도** 최대화: `count(AB) / (count(A) × count(B))`가 가장 큰 쌍 선택 |
| 시작 단위 | UTF-8 바이트 (256개) | 유니코드 문자 |
| 한국어 저장 | 특수 유니코드로 인코딩 → 깨져 보임 | 그대로 저장 → 읽을 수 있음 |
| 어절 경계 표시 | `▁` (공백 바이트 Ġ) | `##` prefix |
| 산출물 파일 | `vocab.json` + `merges.txt` | `vocab.txt` (한 파일) |
| 대표 모델 | GPT-2, GPT-3, RoBERTa | BERT, ELECTRA, KoBERT |

> 💡 **우도 기반 병합의 의미**  
> BPE: "얼마나 자주 붙어 나오는가" (절대 빈도)  
> WordPiece: "각각 따로 나올 때보다 같이 나올 때가 얼마나 더 많은가" (상대 빈도)  
> → 의미 있는 단위로 뭉칠 가능성이 높은 쌍을 우선 병합합니다.

---

> 📌 **Explain 포인트 ①** : `BertWordPieceTokenizer` 파라미터를 설명하세요.
> - `lowercase=False` : 대소문자를 그대로 유지합니다.
> - `special_tokens` : BERT에서 사용하는 5가지 특수 토큰. 각 역할은 아래 표를 참고하세요.

| 특수 토큰 | 인덱스 | 역할 |
|----------|-------|------|
| `[PAD]` | 0 | 패딩. 문장 길이를 맞추기 위한 더미 토큰 |
| `[UNK]` | 1 | 미등록 토큰. 어휘 집합에 없는 서브워드를 대체 |
| `[CLS]` | 2 | 문장 시작. 전체 의미를 집약 (분류 태스크에 활용) |
| `[SEP]` | 3 | 문장 끝/구분. 두 문장 입력 시 경계 역할 |
| `[MASK]` | 4 | 마스킹. MLM(Masked Language Model) 학습 시 사용 |


In [ ]:
# WordPiece 어휘 집합 구축  ← Explain 핵심 셀 ①
#
# BertWordPieceTokenizer 초기화 파라미터:
#   lowercase=False    : 대소문자 구분 유지 (한국어 권장)
#   strip_accents=False: 악센트 부호 유지
#
# .train() 주요 파라미터:
#   files              : 학습 말뭉치 파일 경로 리스트
#   vocab_size         : 최종 어휘 집합 크기
#   min_frequency      : 병합 후보 최소 등장 빈도
#   wordpieces_prefix  : 어절 내 서브워드 구분자 (BERT 표준: "##")
#   special_tokens     : 어휘 집합 앞자리에 예약할 특수 토큰 5종
from tokenizers import BertWordPieceTokenizer

wp_tokenizer = BertWordPieceTokenizer(
    lowercase=False,
    strip_accents=False,
)

print("🔄 WordPiece 토크나이저 학습 중... (약 1~2분 소요)")
wp_tokenizer.train(
    files=[TRAIN_PATH, TEST_PATH],
    vocab_size=10000,
    min_frequency=2,
    wordpieces_prefix="##",
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
)

wp_tokenizer.save_model(WORDPIECE_DIR)

print(f"\n✅ WordPiece 토크나이저 저장 완료: {WORDPIECE_DIR}")
for fname in sorted(os.listdir(WORDPIECE_DIR)):
    fpath = os.path.join(WORDPIECE_DIR, fname)
    print(f"   📄 {fname}  ({os.path.getsize(fpath):,} bytes)")


🔄 WordPiece 토크나이저 학습 중... (약 1~2분 소요)

✅ WordPiece 토크나이저 저장 완료: /gdrive/My Drive/nlpbook/wordpiece
   📄 vocab.txt  (84,790 bytes)


---

### 🔍 `vocab.txt` 구조 이해하기

WordPiece의 결과물인 `vocab.txt`는 **한 파일로 완결**됩니다.  
BPE의 `vocab.json`+`merges.txt` 두 파일과 달리, 병합 규칙이 `vocab.txt` 안에 내포됩니다.

> 💡 **한국어가 깨지지 않는 이유**  
> BPE는 바이트 단위로 시작하여 특수 유니코드로 저장 → Part 1에서 `decode_bbpe_token()` 필요  
> WordPiece는 문자 단위로 시작하여 한국어를 **그대로** 저장 → 별도 변환 함수 불필요

---

> 📌 **Explain 포인트 ②** : `vocab.txt` 내용을 설명하세요.
> - 줄 번호가 곧 토큰 인덱스인 이유
> - `##`으로 시작하는 토큰이 어떤 경우에 만들어지는가
> - BPE의 `vocab.json`과 구조상 어떻게 다른가


In [ ]:
# WordPiece 산출물 확인  ← Explain 핵심 셀 ②
#
# vocab.txt 구조:
#   - 한 줄에 토큰 하나 / 줄 번호(0-indexed) = 토큰 인덱스
#   - 처음 5개: 특수 토큰 [PAD]=0, [UNK]=1, [CLS]=2, [SEP]=3, [MASK]=4
#   - 이후: 단일 문자 → 자주 등장하는 단어 → 서브워드 순
#   - ##으로 시작하는 줄: 어절 내 중간 위치의 서브워드 (어절 첫 토큰이 아님)
with open(f"{WORDPIECE_DIR}/vocab.txt", 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"=== vocab.txt ===")
print(f"총 어휘 수: {len(lines):,}개")

print("\n[ 처음 10개 — 특수 토큰 + 초기 단일 문자 ]")
print(f"  {'인덱스':>6}   토큰")
print("  " + "─" * 30)
for i, line in enumerate(lines[:10]):
    print(f"  {i:6d}   {line.rstrip()}")

print("\n[ ## prefix 토큰 예시 — 어절 내 서브워드 ]")
print(f"  {'인덱스':>6}   토큰")
print("  " + "─" * 30)
count = 0
for i, line in enumerate(lines):
    if line.startswith("##"):
        print(f"  {i:6d}   {line.rstrip()}")
        count += 1
        if count >= 10: break

total_subword = sum(1 for l in lines if l.startswith("##"))
print(f"\n  전체 ## prefix 토큰 수: {total_subword:,}개  "
      f"({total_subword/len(lines)*100:.1f}%)")

print("\n[ 마지막 5개 — 자주 등장하는 긴 서브워드 ]")
for i, line in enumerate(lines[-5:], start=len(lines)-5):
    print(f"  {i:6d}   {line.rstrip()}")


=== vocab.txt ===
총 어휘 수: 10,000개

[ 처음 10개 — 특수 토큰 + 초기 단일 문자 ]
     인덱스   토큰
  ──────────────────────────────
       0   [PAD]
       1   [UNK]
       2   [CLS]
       3   [SEP]
       4   [MASK]
       5   !
       6   "
       7   %
       8   &
       9   '

[ ## prefix 토큰 예시 — 어절 내 서브워드 ]
     인덱스   토큰
  ──────────────────────────────
    1005   ##부
    1006   ##분
    1007   ##의
    1008   ##사
    1009   ##람
    1010   ##들
    1011   ##은
    1012   ##시
    1013   ##도
    1014   ##눈

  전체 ## prefix 토큰 수: 3,533개  (35.3%)

[ 마지막 5개 — 자주 등장하는 긴 서브워드 ]
    9995   에일리언
    9996   99
    9997   very
    9998   ㅠㅠㅠㅠㅠ
    9999   간간히


In [ ]:
# WordPiece 동작 확인 — ## prefix 원리 이해
#
# WordPiece 토큰화 절차:
#   1. 공백으로 어절 분리
#   2. 각 어절을 vocab.txt 기준으로 최장 일치(longest match)로 서브워드 분리
#   3. 어절의 첫 서브워드: ## 없음 (새 어절 시작)
#      이후 서브워드: ## prefix 부착 (앞 토큰과 이어짐을 표시)
#   4. 어절 전체가 어휘 집합에 없으면: [UNK] 처리

sample_words = [
    "짜증나네요",    # 예상: 짜증나 + ##네요
    "초딩영화줄",    # 예상: 초딩 + ##영화 + ##줄
    "오버연기조차",  # 예상: 오버 + ##연기 + ##조차
    "가볍지",        # 예상: 가볍 + ##지
]

print("=== WordPiece ## prefix 동작 예시 ===")
print("어절 내 위치에 따라 ## prefix 유무가 결정됩니다.")
print()
print(f"  {'입력 어절':15}   {'토큰화 결과'}")
print("  " + "─" * 50)
for word in sample_words:
    result = wp_tokenizer.encode(word)
    print(f"  {word:15}   {result.tokens}")


=== WordPiece ## prefix 동작 예시 ===
어절 내 위치에 따라 ## prefix 유무가 결정됩니다.

  입력 어절             토큰화 결과
  ──────────────────────────────────────────────────
  짜증나네요             ['짜증나', '##네요']
  초딩영화줄             ['초딩', '##영화', '##줄']
  오버연기조차            ['오버', '##연기', '##조차']
  가볍지               ['가볍', '##지']


---
## 🔬 심화: BPE vs WordPiece 병합 기준 정량 분석

> 교재 수식 2-1 직접 구현하기

이 섹션에서는 BPE와 WordPiece가 **같은 말뭉치에서 서로 다른 쌍을 병합하는 이유**를  
실제 NSMC 데이터로 수치를 계산하여 직접 확인합니다.

---

### 📐 두 알고리즘의 병합 점수 공식

**BPE 점수** — 단순 빈도(등장 횟수)
```
score_BPE(a, b)  =  count(ab)
```

**WordPiece 점수** — 우도(likelihood) 기반 (교재 수식 2-1)
```
              count(ab) / n
score_WP(a, b) = ─────────────────
              count(a)/n × count(b)/n

             count(ab)
           = ─────────────────
             count(a) × count(b)
```

> 💡 **핵심 직관**  
> - BPE: `ab`가 **절대적으로 많이** 나타나는 쌍을 선택  
> - WordPiece: `a`와 `b`가 **독립적으로 등장할 때보다 함께 등장할 때가 훨씬 많은** 쌍을 선택  
>  
> 예시로 생각해 봅시다.  
> `이` 와 `다` 는 각자도 엄청 자주 등장하고, 붙어서도 자주 등장합니다.  
> → BPE는 이 쌍을 높이 평가하지만, WordPiece는 "어차피 각자도 많이 나오니까" 낮게 평가합니다.  
>  
> 반면 `스럽` 과 `다` 는 각자는 그렇게 흔하지 않지만, 붙어서는 거의 항상 나옵니다.  
> → WordPiece는 이 쌍을 높이 평가합니다.


In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 1: 말뭉치에서 문자 단위 유니그램·바이그램 빈도 집계
#
# 토크나이저 학습 첫 단계를 재현합니다.
# 실제 BPE/WordPiece는 어절(공백 분리) 내에서 문자 단위로 시작하여
# 가장 점수가 높은 인접 쌍을 하나씩 병합해 나갑니다.
#
# 여기서는 그 초기 상태(모든 문자가 개별 토큰인 상태)에서
# BPE 점수와 WordPiece 점수를 직접 계산하여 비교합니다.
# ──────────────────────────────────────────────────────────────
from collections import Counter
import math

print("📊 NSMC train.txt에서 문자 빈도를 집계합니다...")

unigram_counter = Counter()   # 단일 문자 빈도: count(a)
bigram_counter  = Counter()   # 인접 문자 쌍 빈도: count(ab)
total_chars     = 0

with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 30000: break   # 속도를 위해 3만 문장만 사용
        for word in line.split():
            chars = list(word)
            # 유니그램 집계
            unigram_counter.update(chars)
            total_chars += len(chars)
            # 바이그램 집계 (어절 경계를 넘지 않음)
            for j in range(len(chars) - 1):
                bigram_counter[(chars[j], chars[j+1])] += 1

n = total_chars

print(f"✅ 집계 완료")
print(f"   총 문자 수(n)      : {n:,}")
print(f"   유니그램 종류     : {len(unigram_counter):,}개")
print(f"   바이그램 종류     : {len(bigram_counter):,}개")
print(f"\n  [ 가장 많이 등장한 유니그램 Top 10 ]")
print(f"  {'문자':>6}   {'빈도':>10}")
print("  " + "─" * 25)
for ch, cnt in unigram_counter.most_common(10):
    print(f"  {ch:>6}   {cnt:>10,}")


📊 NSMC train.txt에서 문자 빈도를 집계합니다...
✅ 집계 완료
   총 문자 수(n)      : 864,761
   유니그램 종류     : 2,075개
   바이그램 종류     : 58,348개

  [ 가장 많이 등장한 유니그램 Top 10 ]
      문자           빈도
  ─────────────────────────
       .       48,223
       이       27,806
       다       22,008
       는       18,286
       고       14,414
       지       13,455
       화       13,267
       영       12,990
       가       11,495
       하       11,461


In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 2: BPE 점수 & WordPiece 점수 계산
#
# BPE 점수   = count(ab)                       ← 단순 빈도
# WP  점수   = count(ab) / (count(a) × count(b)) ← 우도 기반
# ──────────────────────────────────────────────────────────────

scores = []
for (a, b), cnt_ab in bigram_counter.items():
    cnt_a = unigram_counter[a]
    cnt_b = unigram_counter[b]

    score_bpe = cnt_ab                             # BPE: 단순 빈도
    score_wp  = cnt_ab / (cnt_a * cnt_b)           # WordPiece: 우도 기반

    scores.append({
        "pair"     : f"{a} + {b}",
        "a"        : a,
        "b"        : b,
        "cnt_ab"   : cnt_ab,
        "cnt_a"    : cnt_a,
        "cnt_b"    : cnt_b,
        "score_bpe": score_bpe,
        "score_wp" : score_wp,
    })

# ── BPE 기준 상위 20개 ────────────────────────────────────────
bpe_top = sorted(scores, key=lambda x: x["score_bpe"], reverse=True)[:20]
wp_top  = sorted(scores, key=lambda x: x["score_wp"],  reverse=True)[:20]

print("=" * 70)
print("  BPE 기준 상위 20개 쌍  (단순 빈도: count(ab) 높은 순)")
print("=" * 70)
print(f"  {'쌍':>8}   {'count(ab)':>10}   {'count(a)':>10}   {'count(b)':>10}   {'WP 점수':>12}")
print("  " + "─" * 62)
for r in bpe_top:
    print(f"  {r['pair']:>8}   {r['cnt_ab']:>10,}   {r['cnt_a']:>10,}   {r['cnt_b']:>10,}   {r['score_wp']:>12.6f}")


  BPE 기준 상위 20개 쌍  (단순 빈도: count(ab) 높은 순)
         쌍    count(ab)     count(a)     count(b)          WP 점수
  ──────────────────────────────────────────────────────────────
     . + .       21,575       48,223       48,223       0.000009
     영 + 화       11,882       12,990       13,267       0.000069
     다 + .        8,011       22,008       48,223       0.000008
     ㅋ + ㅋ        4,430        6,792        6,792       0.000096
     ! + !        2,806        6,263        6,263       0.000072
     니 + 다        2,605        6,020       22,008       0.000020
     는 + 데        2,584       18,286        4,634       0.000030
     재 + 미        2,459        5,499        4,680       0.000096
     너 + 무        2,400        2,624        4,538       0.000202
     하 + 고        2,213       11,461       14,414       0.000013
     정 + 말        2,019        4,642        3,962       0.000110
     하 + 는        1,933       11,461       18,286       0.000009
     지 + 만        1,913       13,455        7,6

In [ ]:
# ── WordPiece 기준 상위 20개 ─────────────────────────────────
print("=" * 70)
print("  WordPiece 기준 상위 20개 쌍  (우도: count(ab)/(count(a)×count(b)) 높은 순)")
print("=" * 70)
print(f"  {'쌍':>8}   {'WP 점수':>12}   {'count(ab)':>10}   {'count(a)':>10}   {'count(b)':>10}")
print("  " + "─" * 62)
for r in wp_top:
    print(f"  {r['pair']:>8}   {r['score_wp']:>12.6f}   {r['cnt_ab']:>10,}   {r['cnt_a']:>10,}   {r['cnt_b']:>10,}")


  WordPiece 기준 상위 20개 쌍  (우도: count(ab)/(count(a)×count(b)) 높은 순)
         쌍          WP 점수    count(ab)     count(a)     count(b)
  ──────────────────────────────────────────────────────────────
     必 + 要       1.000000            1            1            1
     秀 + 作       1.000000            1            1            1
     ´ + ∇       1.000000            1            1            1
     ∇ + ｀       1.000000            1            1            1
     最 + 高       1.000000            1            1            1
     自 + 立       1.000000            1            1            1
     會 + 者       1.000000            1            1            1
     者 + 定       1.000000            1            1            1
     定 + 離       1.000000            1            1            1
     漁 + 火       1.000000            1            1            1
     間 + 道       1.000000            1            1            1
     日 + 本       1.000000            1            1            1
     低 + 力       1.00000

---

### 📊 두 점수 체계에서 순위가 역전되는 쌍 찾기

BPE와 WordPiece가 **실제로 다른 선택을 하는** 쌍을 구체적으로 찾아봅니다.

> 📌 **강사 설명 포인트**  
>  
> **BPE 상위 ≠ WordPiece 상위** 인 쌍이 생기는 이유:  
> - BPE 상위에 오지만 WP 점수가 낮은 쌍  
>   → `count(a)`도 크고 `count(b)`도 크다  
>   → 즉, 각자가 매우 흔한 글자들이라 "함께 나올 이유가 약하다"고 판단  
>   → 예: `이 + 다`, `하 + 다`, `어 + 다` 처럼 자주 등장하는 종결 형태  
>  
> - WP 상위에 오지만 BPE 점수가 낮은 쌍  
>   → `count(ab)`는 크지 않지만, `count(a)`, `count(b)` 각각도 작다  
>   → 즉, 이 두 글자는 서로 떨어져서는 잘 안 나오는데 붙으면 자주 나온다  
>   → 예: 특정 외래어, 고유명사의 연속 글자들


In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 3: 두 점수 체계에서 순위가 역전되는 쌍 분석
# ──────────────────────────────────────────────────────────────

# BPE 순위 딕셔너리
bpe_rank = {r["pair"]: i+1 for i, r in
            enumerate(sorted(scores, key=lambda x: x["score_bpe"], reverse=True))}
wp_rank  = {r["pair"]: i+1 for i, r in
            enumerate(sorted(scores, key=lambda x: x["score_wp"],  reverse=True))}

# 역전 폭이 큰 쌍 추출
divergence = []
for s in scores:
    if s["cnt_ab"] < 50: continue   # 너무 희귀한 쌍은 제외
    br = bpe_rank[s["pair"]]
    wr = wp_rank[s["pair"]]
    diff = wr - br   # 양수: WP 순위가 낮다 (BPE가 더 선호) / 음수: WP 순위가 높다
    divergence.append((diff, br, wr, s))

divergence.sort(key=lambda x: x[0], reverse=True)

# ── BPE는 선호하지만 WordPiece는 상대적으로 낮게 평가하는 쌍 ──
print("=" * 75)
print("  [BPE 선호 > WP 선호] — 빈도는 높지만 각자도 흔해서 WP 점수가 낮은 쌍")
print("  (BPE: 병합하고 싶다 | WordPiece: 굳이?)")
print("=" * 75)
print(f"  {'쌍':>8}   {'BPE순위':>8}   {'WP순위':>8}   {'순위차':>8}   "
      f"{'count(ab)':>10}   {'count(a)':>8}   {'count(b)':>8}   {'WP점수':>10}")
print("  " + "─" * 78)
for diff, br, wr, s in divergence[:15]:
    print(f"  {s['pair']:>8}   {br:>8,}   {wr:>8,}   {diff:>+8,}   "
          f"{s['cnt_ab']:>10,}   {s['cnt_a']:>8,}   {s['cnt_b']:>8,}   {s['score_wp']:>10.6f}")

print()
print("=" * 75)
print("  [WP 선호 > BPE 선호] — 빈도는 낮지만 '함께 등장할 이유'가 강한 쌍")
print("  (WordPiece: 병합해야 한다 | BPE: 별로 안 보이는데?)")
print("=" * 75)
print(f"  {'쌍':>8}   {'BPE순위':>8}   {'WP순위':>8}   {'순위차':>8}   "
      f"{'count(ab)':>10}   {'count(a)':>8}   {'count(b)':>8}   {'WP점수':>10}")
print("  " + "─" * 78)
for diff, br, wr, s in divergence[-15:][::-1]:
    print(f"  {s['pair']:>8}   {br:>8,}   {wr:>8,}   {diff:>+8,}   "
          f"{s['cnt_ab']:>10,}   {s['cnt_a']:>8,}   {s['cnt_b']:>8,}   {s['score_wp']:>10.6f}")


  [BPE 선호 > WP 선호] — 빈도는 높지만 각자도 흔해서 WP 점수가 낮은 쌍
  (BPE: 병합하고 싶다 | WordPiece: 굳이?)
         쌍      BPE순위       WP순위        순위차    count(ab)   count(a)   count(b)         WP점수
  ──────────────────────────────────────────────────────────────────────────────
     . + 다      1,237     56,776    +55,539           76     48,223     22,008     0.000000
     이 + .        755     56,137    +55,382          124     27,806     48,223     0.000000
     . + 지      1,620     56,226    +54,606           58     48,223     13,455     0.000000
     다 + 이      1,621     56,060    +54,439           58     22,008     27,806     0.000000
     하 + .      1,656     55,765    +54,109           57     11,461     48,223     0.000000
     . + 하      1,441     55,312    +53,871           65     48,223     11,461     0.000000
     . + 나      1,554     54,545    +52,991           60     48,223      8,845     0.000000
     영 + 이      1,729     54,137    +52,408           55     12,990     27,806     0.000000
     . +

---

### 📈 구체적인 사례로 이해하기

위 분석에서 찾은 흥미로운 쌍들을 **왜 점수가 다르게 나오는지** 수식으로 직접 풀어봅니다.


In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 4: 특정 쌍에 대한 BPE / WordPiece 점수 계산 상세 설명
#
# ↓ 위 분석 결과에서 흥미로운 쌍을 골라 직접 넣어 보세요.
#   예: BPE는 좋아하지만 WP는 별로인 쌍, 반대 사례 등
# ──────────────────────────────────────────────────────────────

def explain_pair(a, b):
    """
    두 문자 a, b에 대해 BPE 점수와 WordPiece 점수를
    수식 단계별로 출력합니다.
    """
    cnt_a  = unigram_counter.get(a, 0)
    cnt_b  = unigram_counter.get(b, 0)
    cnt_ab = bigram_counter.get((a, b), 0)

    if cnt_a == 0 or cnt_b == 0:
        print(f"'{a}' 또는 '{b}'가 말뭉치에 없습니다."); return

    score_bpe = cnt_ab
    score_wp  = cnt_ab / (cnt_a * cnt_b)

    bpe_r = bpe_rank.get(f"{a} + {b}", "─")
    wp_r  = wp_rank.get(f"{a} + {b}", "─")

    print(f"┌─────────────────────────────────────────────┐")
    print(f"│  쌍: '{a}' + '{b}'                           │")
    print(f"├─────────────────────────────────────────────┤")
    print(f"│  count(a)  = count('{a}')  = {cnt_a:>10,}  │")
    print(f"│  count(b)  = count('{b}')  = {cnt_b:>10,}  │")
    print(f"│  count(ab) = count('{a}{b}') = {cnt_ab:>10,}  │")
    print(f"│  n(총 문자수)             = {n:>10,}  │")
    print(f"├─────────────────────────────────────────────┤")
    print(f"│  BPE  점수 = count(ab) = {score_bpe:>14,}  │")
    print(f"│  BPE  순위 = {str(bpe_r):>30}  │")
    print(f"├─────────────────────────────────────────────┤")
    print(f"│  WP 점수 공식:                               │")
    print(f"│    count(ab) / (count(a) × count(b))         │")
    print(f"│  = {cnt_ab:,} / ({cnt_a:,} × {cnt_b:,})  │")
    print(f"│  = {score_wp:.8f}                    │")
    print(f"│  WP   순위 = {str(wp_r):>30}  │")
    print(f"└─────────────────────────────────────────────┘")

    # 해석
    if isinstance(bpe_r, int) and isinstance(wp_r, int):
        if wp_r > bpe_r:
            print(f"  → BPE 순위({bpe_r:,}위) < WP 순위({wp_r:,}위)")
            print(f"    '{a}'도 자주, '{b}'도 자주 등장하므로")
            print(f"    함께 나오는 것이 '우연'에 가깝다고 WP가 판단합니다.")
        elif wp_r < bpe_r:
            print(f"  → WP 순위({wp_r:,}위) < BPE 순위({bpe_r:,}위)")
            print(f"    '{a}'과 '{b}'가 각자는 드물지만 붙으면 자주 등장합니다.")
            print(f"    '함께 등장할 이유'가 강하다고 WP가 판단합니다.")
    print()

# ── 사례 분석 ────────────────────────────────────────────────
# (1) BPE가 선호하고 WP는 상대적으로 낮게 보는 쌍
#     → 두 글자 각자가 매우 흔한 경우
print("=" * 50)
print("  사례 ① BPE ↑  WP ↓  — 흔한 글자 조합")
print("=" * 50)
for a, b in [("이", "다"), ("하", "다"), ("었", "다")]:
    explain_pair(a, b)

# (2) WP가 선호하고 BPE는 상대적으로 낮게 보는 쌍
#     → 두 글자 각자는 드물지만 붙으면 자주 등장하는 경우
print("=" * 50)
print("  사례 ② WP ↑  BPE ↓  — 결합력이 강한 쌍")
print("=" * 50)
# WP 상위 쌍 중에서 BPE 순위가 낮은 것을 자동 선택
wp_preferred = [(diff, br, wr, s) for diff, br, wr, s in divergence
                if diff < -200 and s["cnt_ab"] >= 100][-6:]
for _, _, _, s in wp_preferred[::-1][:3]:
    explain_pair(s["a"], s["b"])


  사례 ① BPE ↑  WP ↓  — 흔한 글자 조합
┌─────────────────────────────────────────────┐
│  쌍: '이' + '다'                           │
├─────────────────────────────────────────────┤
│  count(a)  = count('이')  =     27,806  │
│  count(b)  = count('다')  =     22,008  │
│  count(ab) = count('이다') =      1,141  │
│  n(총 문자수)             =    864,761  │
├─────────────────────────────────────────────┤
│  BPE  점수 = count(ab) =          1,141  │
│  BPE  순위 =                             30  │
├─────────────────────────────────────────────┤
│  WP 점수 공식:                               │
│    count(ab) / (count(a) × count(b))         │
│  = 1,141 / (27,806 × 22,008)  │
│  = 0.00000186                    │
│  WP   순위 =                          31494  │
└─────────────────────────────────────────────┘
  → BPE 순위(30위) < WP 순위(31,494위)
    '이'도 자주, '다'도 자주 등장하므로
    함께 나오는 것이 '우연'에 가깝다고 WP가 판단합니다.

┌─────────────────────────────────────────────┐
│  쌍: '하' + '다'                           │
├────────────────────────

---

### 📊 시각적 비교: 점수 산포도

같은 말뭉치에서 BPE 점수(빈도)와 WordPiece 점수(우도)의 관계를 산포도로 확인합니다.

> **기울기 없이 넓게 퍼져 있을수록** 두 알고리즘이 서로 다른 쌍을 선택한다는 의미입니다.


In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 5: BPE 점수 vs WordPiece 점수 산포도 (텍스트 기반)
#
# matplotlib 대신 텍스트로 구현 — Colab에서 항상 안정적으로 동작합니다.
# ──────────────────────────────────────────────────────────────
import math

# 분석 대상: count(ab) >= 100 인 바이그램만
filtered = [s for s in scores if s["cnt_ab"] >= 100]

# 로그 스케일로 정규화 (산포도 표시용)
bpe_vals = [math.log10(s["cnt_ab"])            for s in filtered]
wp_vals  = [math.log10(s["score_wp"] + 1e-12)  for s in filtered]

# 정규화 함수 (0~1)
def norm(vals):
    mn, mx = min(vals), max(vals)
    return [(v - mn) / (mx - mn) for v in vals]

bpe_norm = norm(bpe_vals)
wp_norm  = norm(wp_vals)

# 텍스트 그리드 (40×20)
W, H = 50, 20
grid = [["·"] * W for _ in range(H)]

for bv, wv, s in zip(bpe_norm, wp_norm, filtered):
    x = min(int(bv * (W-1)), W-1)
    y = min(int(wv * (H-1)), H-1)
    y_flip = (H-1) - y   # y축 반전 (위로 갈수록 큰 값)
    ch = grid[y_flip][x]
    if ch == "·":
        grid[y_flip][x] = "●"
    else:
        grid[y_flip][x] = "◉"  # 밀집 구역

print("=" * (W + 10))
print(f"  BPE 점수(x축: →) vs WordPiece 점수(y축: ↑)  [log 스케일, n={len(filtered)}쌍]")
print("=" * (W + 10))
print(f"  WP↑  │")
for row_i, row in enumerate(grid):
    label = "  높음 │" if row_i == 0 else             "  낮음 │" if row_i == H-1 else             "       │"
    print(label + "".join(row))
print("       └" + "─" * W)
print("        낮음" + " " * (W - 10) + "높음  BPE→")
print()
print("  ● : 해당 점수 범위의 바이그램  ◉ : 2개 이상 중첩")
print()
# 상관계수 계산
def pearson(x, y):
    n = len(x)
    mx, my = sum(x)/n, sum(y)/n
    num = sum((xi-mx)*(yi-my) for xi, yi in zip(x, y))
    den = (sum((xi-mx)**2 for xi in x) * sum((yi-my)**2 for yi in y)) ** 0.5
    return num / den if den else 0

r = pearson(bpe_norm, wp_norm)
print(f"  Pearson 상관계수(log 스케일): r = {r:.4f}")
if r > 0.7:
    print("  → 두 점수가 비교적 일치하는 편입니다.")
elif r > 0.4:
    print("  → 두 점수가 어느 정도 다른 방향을 가리킵니다.")
else:
    print("  → 두 점수 체계가 매우 다른 쌍을 선택합니다.")


  BPE 점수(x축: →) vs WordPiece 점수(y축: ↑)  [log 스케일, n=943쌍]
  WP↑  │
  높음 │····●·············································
       │·●················································
       │●●●·····●·●·······································
       │◉◉●◉●···●·······●·································
       │◉◉●··●●●◉·····◉···································
       │◉◉◉●·●◉·●◉●··●·◉·◉····●●··························
       │◉●◉◉◉·◉◉●◉◉●··◉●●·●◉●●·●·●●·●·····················
       │◉◉◉●◉◉◉◉◉◉◉●◉●●●·●●·◉◉●····●······················
       │◉◉◉◉◉◉◉◉◉◉◉·◉◉●◉●●·●··●···◉··●●···●········●······
       │◉◉◉◉◉◉◉◉◉◉◉◉◉◉◉●◉·◉◉·●·●·····●····················
       │◉◉◉◉◉◉◉◉◉●◉◉●◉◉◉◉●◉◉●●●●·●●··●····················
       │◉◉◉◉◉◉◉◉◉◉◉◉◉◉◉◉●●·◉·◉··●···●·····················
       │◉◉◉◉◉◉◉◉◉◉◉◉◉◉◉◉◉◉●·◉●●···●●···········●·········●
       │◉◉◉◉◉◉◉◉◉◉◉◉◉◉●◉·◉●◉·●····························
       │◉◉◉◉◉◉◉◉◉◉●◉·◉◉◉···●··●··●························
       │◉◉◉◉◉·◉◉●·●·◉●···◉························

---

### 🔄 한 단계 병합 시뮬레이션

BPE와 WordPiece가 각각 **1번의 병합**을 수행한다면 어떤 쌍을 선택하는지 직접 시뮬레이션합니다.  
이것이 두 알고리즘이 최종적으로 서로 다른 어휘 집합을 만들어내는 근본 이유입니다.


In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 6: 1-step 병합 시뮬레이션
#
# 같은 초기 상태(문자 단위)에서 시작하여
# BPE와 WordPiece가 각각 처음 선택하는 10개의 병합 쌍을 비교합니다.
# ──────────────────────────────────────────────────────────────

print("=" * 68)
print("  같은 말뭉치에서 BPE와 WordPiece가 처음 선택하는 병합 쌍 Top 10")
print("=" * 68)
print()

bpe_sorted = sorted(scores, key=lambda x: x["score_bpe"], reverse=True)
wp_sorted  = sorted(scores, key=lambda x: x["score_wp"],  reverse=True)

print(f"  {'순위':>4}   {'BPE 선택':>10}   {'BPE 빈도':>10}   │   "
      f"{'WP 선택':>10}   {'WP 점수':>14}")
print("  " + "─" * 68)

for rank, (bpe, wp) in enumerate(zip(bpe_sorted[:10], wp_sorted[:10]), 1):
    same_marker = " ← 동일" if bpe["pair"] == wp["pair"] else ""
    print(f"  {rank:>4}   {bpe['pair']:>10}   {bpe['cnt_ab']:>10,}   │   "
          f"{wp['pair']:>10}   {wp['score_wp']:>14.8f}{same_marker}")

# 공통 쌍 확인
bpe_top10_pairs = {s["pair"] for s in bpe_sorted[:10]}
wp_top10_pairs  = {s["pair"] for s in wp_sorted[:10]}
common          = bpe_top10_pairs & wp_top10_pairs

print()
print(f"  Top 10 중 공통으로 선택된 쌍: {len(common)}개  {common if common else '(없음)'}")
print()
print("  📌 해석:")
print("  - BPE는 말뭉치에서 절대적으로 많이 붙어 나오는 쌍을 선택합니다.")
print("  - WordPiece는 개별 등장 빈도 대비 함께 등장하는 비율이 높은 쌍을 선택합니다.")
print("  - 두 알고리즘의 Top 10이 다를수록 최종 어휘 집합도 달라집니다.")
print("  - 이 차이가 누적되어 WordPiece가 더 '의미 있는' 서브워드를 만드는 이유입니다.")


  같은 말뭉치에서 BPE와 WordPiece가 처음 선택하는 병합 쌍 Top 10

    순위       BPE 선택       BPE 빈도   │        WP 선택            WP 점수
  ────────────────────────────────────────────────────────────────────
     1        . + .       21,575   │        必 + 要       1.00000000
     2        영 + 화       11,882   │        秀 + 作       1.00000000
     3        다 + .        8,011   │        ´ + ∇       1.00000000
     4        ㅋ + ㅋ        4,430   │        ∇ + ｀       1.00000000
     5        ! + !        2,806   │        最 + 高       1.00000000
     6        니 + 다        2,605   │        自 + 立       1.00000000
     7        는 + 데        2,584   │        會 + 者       1.00000000
     8        재 + 미        2,459   │        者 + 定       1.00000000
     9        너 + 무        2,400   │        定 + 離       1.00000000
    10        하 + 고        2,213   │        漁 + 火       1.00000000

  Top 10 중 공통으로 선택된 쌍: 0개  (없음)

  📌 해석:
  - BPE는 말뭉치에서 절대적으로 많이 붙어 나오는 쌍을 선택합니다.
  - WordPiece는 개별 등장 빈도 대비 함께 등장하는 비율이 높은 쌍을 선택합니다.
  - 두 알고리

---

### 💡 정리: 왜 WordPiece 점수가 더 '의미 있는' 서브워드를 만드는가?

| 상황 | BPE 판단 | WordPiece 판단 |
|------|---------|--------------|
| `이 + 다` (각자 매우 흔함) | 자주 붙으니 병합! | 각자도 흔하니 굳이? |
| `스럽 + 다` (각자는 드뭄) | 별로 안 보이니 패스 | 거의 항상 붙으니 병합! |
| `딥 + 러` (전문 용어 일부) | 별로 안 보이니 패스 | 독립 등장 거의 없으니 병합! |

> 🔑 **핵심**: WordPiece는 "두 토큰이 따로 등장할 때보다 같이 등장할 때가 얼마나 더 많은가"를 봅니다.  
> 이 측도는 통계학의 **점별 상호 정보량(Pointwise Mutual Information, PMI)**과 본질적으로 같습니다.  
>  
> PMI(a, b) = log[ P(a,b) / (P(a) × P(b)) ]  
>  
> WordPiece 점수의 분자 count(ab)/분모 count(a)×count(b) 는 PMI의 확률 비율 부분에 해당합니다.


---
## 4단계: BERT 입력값 만들기

### 📐 BERT 입력값의 구성

| 입력값 | GPT | BERT |
|--------|-----|------|
| `input_ids` | 토큰 인덱스 배열 | 동일. 단, 앞에 `[CLS]`, 뒤에 `[SEP]` 자동 추가 |
| `attention_mask` | 실제=1, 패딩=0 | 동일 |
| `token_type_ids` | 없음 | **BERT 고유**. 첫 문장=0, 두 번째 문장=1 |

> 💡 **`[CLS]`와 `[SEP]`를 자동으로 추가하는 이유**  
> BERT는 학습 시 항상 이 두 토큰을 사용하도록 설계되었습니다.  
> `[CLS]` : 문장 전체를 대표. 분류 태스크에서 이 토큰의 출력 벡터를 분류기에 입력합니다.  
> `[SEP]` : 문장의 끝 또는 두 문장 사이의 경계를 알립니다.

> 💡 **`token_type_ids`가 필요한 이유**  
> BERT는 두 문장을 동시에 입력받아 문장 간 관계를 학습하는 NSP(Next Sentence Prediction)를 수행합니다.  
> `token_type_ids`로 각 토큰이 어느 문장에 속하는지 모델에 알려야 올바른 문장 관계를 학습할 수 있습니다.

---

> 📌 **Explain 포인트 ③** : `BertTokenizer.from_pretrained()` 파라미터를 설명하세요.
> - `do_lower_case=False` 를 학습 시 `lowercase=False`와 반드시 일치시켜야 하는 이유는?  
>   → 불일치 시 학습에서 보지 못한 토큰이 `[UNK]`으로 처리되어 성능이 크게 저하됩니다.


In [ ]:
# BERT 토크나이저 로드
#
# BertTokenizer.from_pretrained(경로)
#   - 지정된 경로에서 vocab.txt 를 읽어 토크나이저를 초기화합니다.
#
# do_lower_case=False
#   - 학습 시 lowercase=False 로 학습했으므로 반드시 일치시켜야 합니다.
from transformers import BertTokenizer

tokenizer_bert = BertTokenizer.from_pretrained(
    WORDPIECE_DIR,
    do_lower_case=False,
)

print(f"✅ BERT 토크나이저 로드 완료")
print(f"   어휘 집합 크기 : {len(tokenizer_bert):,}개")
print()
print(f"   특수 토큰 인덱스:")
print(f"     [PAD]  = {tokenizer_bert.pad_token_id}  ← 패딩 토큰")
print(f"     [UNK]  = {tokenizer_bert.unk_token_id}  ← 미등록 토큰")
print(f"     [CLS]  = {tokenizer_bert.cls_token_id}  ← 문장 시작, 분류 태스크용")
print(f"     [SEP]  = {tokenizer_bert.sep_token_id}  ← 문장 끝/구분")
print(f"     [MASK] = {tokenizer_bert.mask_token_id}  ← MLM 학습용")


✅ BERT 토크나이저 로드 완료
   어휘 집합 크기 : 10,000개

   특수 토큰 인덱스:
     [PAD]  = 0  ← 패딩 토큰
     [UNK]  = 1  ← 미등록 토큰
     [CLS]  = 2  ← 문장 시작, 분류 태스크용
     [SEP]  = 3  ← 문장 끝/구분
     [MASK] = 4  ← MLM 학습용


In [ ]:
# BERT 토크나이저 — 예시 문장 토큰화  ← Explain 핵심 셀 ③
#
# ## prefix 해석:
#   ## 없는 토큰: 어절의 첫 번째 서브워드 (새 어절 시작)
#   ## 있는 토큰: 앞 토큰에 이어지는 서브워드 (어절 계속)

sentences = [
    "아 더빙.. 진짜 짜증나네요 목소리",
    "흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나",
    "별루 였다..",
]

print("=== BERT 토크나이저 토큰화 결과 ===")
print("## 없음: 새 어절 시작 | [##토큰]: 앞 토큰에 이어지는 서브워드")
for i, sent in enumerate(sentences, 1):
    tokens = tokenizer_bert.tokenize(sent)
    highlighted = [f"[{t}]" if t.startswith("##") else t for t in tokens]
    print(f"\n[문장{i}] {sent}")
    print(f"  토큰 수 : {len(tokens)}")
    print(f"  토큰    : {highlighted}")


=== BERT 토크나이저 토큰화 결과 ===
## 없음: 새 어절 시작 | [##토큰]: 앞 토큰에 이어지는 서브워드

[문장1] 아 더빙.. 진짜 짜증나네요 목소리
  토큰 수 : 8
  토큰    : ['아', '더빙', '.', '.', '진짜', '짜증나', '[##네요]', '목소리']

[문장2] 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
  토큰 수 : 20
  토큰    : ['흠', '.', '.', '.', '포스터', '[##보고]', '초딩', '[##영화]', '[##줄]', '.', '.', '.', '.', '오버', '[##연기]', '[##조차]', '가볍', '[##지]', '않', '[##구나]']

[문장3] 별루 였다..
  토큰 수 : 4
  토큰    : ['별루', '였다', '.', '.']


---

> 📌 **Explain 포인트 ④** : 아래 출력에서 `input_ids` 앞뒤 값을 설명하세요.
> - 토큰01이 항상 `2`(=[CLS])인 이유
> - 실제 마지막 토큰 바로 뒤가 `3`(=[SEP])인 이유
> - `token_type_ids`가 모두 `0`인 이유


In [ ]:
# BERT 모델 입력 만들기 — 단일 문장
#
# 내부 동작:
#   tokenize → [CLS] 앞 추가, [SEP] 뒤 추가 → indexing → padding/truncation
#
# max_length=12: 실제 모델에서는 512 사용; 표 표시 편의를 위해 12로 설정

batch_bert = tokenizer_bert(
    sentences,
    padding="max_length",
    max_length=12,
    truncation=True,
)

header = "구분    " + "  ".join([f"토큰{i:02d}" for i in range(1, 13)])
sep    = "─" * 82

print("=== BERT input_ids ===")
print(f"  앞에 [CLS](id={tokenizer_bert.cls_token_id})가 자동 추가됩니다.")
print(f"  뒤에 [SEP](id={tokenizer_bert.sep_token_id})가 자동 추가됩니다.")
print(f"  0 = [PAD] 토큰")
print()
print(header); print(sep)
for i, ids in enumerate(batch_bert['input_ids'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in ids]))

# 문장1 상세 — 인덱스 → 토큰 문자열 역변환
print()
print("[ 문장1 상세: 각 위치의 인덱스와 토큰 ]")
ids1    = batch_bert['input_ids'][0]
tokens1 = tokenizer_bert.convert_ids_to_tokens(ids1)
print(f"  {'위치':>4}   {'id':>6}   토큰")
print("  " + "─" * 35)
for pos, (tid, tok) in enumerate(zip(ids1, tokens1), 1):
    marker = " ← [CLS]" if tok=="[CLS]" else " ← [SEP]" if tok=="[SEP]" else              " ← [PAD]" if tok=="[PAD]" else " ← 서브워드" if tok.startswith("##") else ""
    print(f"  {pos:4d}   {tid:6d}   {tok}{marker}")

print()
print("=== BERT attention_mask ===")
print("  1 = 실제 토큰 ([CLS],[SEP] 포함)  /  0 = 패딩")
print()
print(header); print(sep)
for i, mask in enumerate(batch_bert['attention_mask'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in mask]))

print()
print("=== BERT token_type_ids ===")
print("  단일 문장 입력이므로 모두 0  (두 문장 입력 시 두 번째 문장 위치는 1)")
print()
print(header); print(sep)
for i, ttids in enumerate(batch_bert['token_type_ids'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in ttids]))


=== BERT input_ids ===
  앞에 [CLS](id=2)가 자동 추가됩니다.
  뒤에 [SEP](id=3)가 자동 추가됩니다.
  0 = [PAD] 토큰

구분    토큰01  토큰02  토큰03  토큰04  토큰05  토큰06  토큰07  토큰08  토큰09  토큰10  토큰11  토큰12
──────────────────────────────────────────────────────────────────────────────────
문장1      2    621   2631     16     16   1993   3678   1990   3323      3      0      0
문장2      2    997     16     16     16   2609   2045   2796   1981   1272     16      3
문장3      2   3274   9508     16     16      3      0      0      0      0      0      0

[ 문장1 상세: 각 위치의 인덱스와 토큰 ]
    위치       id   토큰
  ───────────────────────────────────
     1        2   [CLS] ← [CLS]
     2      621   아
     3     2631   더빙
     4       16   .
     5       16   .
     6     1993   진짜
     7     3678   짜증나
     8     1990   ##네요 ← 서브워드
     9     3323   목소리
    10        3   [SEP] ← [SEP]
    11        0   [PAD] ← [PAD]
    12        0   [PAD] ← [PAD]

=== BERT attention_mask ===
  1 = 실제 토큰 ([CLS],[SEP] 포함)  /  0 = 패딩

구분    토큰01  토큰02  토큰0

---

### 📊 두 문장 입력 — `token_type_ids` 실제 동작 확인

BERT의 고유 기능인 두 문장 동시 입력을 직접 실습합니다.

> 📌 **Explain 포인트 ⑤** : 아래 출력 구조를 설명하세요.
> - 입력 형식: `[CLS] 문장A [SEP] 문장B [SEP]`
> - `token_type_ids`: 문장A 영역(`[CLS]` 포함)은 `0`, 문장B 영역(`[SEP]` 포함)은 `1`
> - 이 정보가 없으면 BERT가 NSP를 어떻게 학습할 수 없는가?


In [ ]:
# 두 문장 동시 입력 — token_type_ids 확인
#
# tokenizer_bert(문장A, 문장B) 형태로 쌍을 전달하면
# 내부적으로 "[CLS] 문장A [SEP] 문장B [SEP]" 구조로 구성됩니다.
#
# token_type_ids:
#   첫 번째 세그먼트 ([CLS] 포함 ~ 첫 [SEP] 포함) : 0
#   두 번째 세그먼트 (문장B 시작 ~ 두 번째 [SEP] 포함) : 1

sent_a = "이 영화 재미있었어요"
sent_b = "정말 추천합니다"

pair_input = tokenizer_bert(
    sent_a, sent_b,
    padding="max_length",
    max_length=16,
    truncation=True,
)

ids    = pair_input['input_ids']
ttids  = pair_input['token_type_ids']
mask   = pair_input['attention_mask']
tokens = tokenizer_bert.convert_ids_to_tokens(ids)

print(f"=== 두 문장 동시 입력 ===")
print(f"  문장A: '{sent_a}'")
print(f"  문장B: '{sent_b}'")
print(f"  입력 형식: [CLS] 문장A [SEP] 문장B [SEP]")
print()
print(f"  {'위치':>4}   {'id':>6}   {'type':>6}   {'att':>5}   토큰   ← 세그먼트")
print("  " + "─" * 58)
for pos, (tid, tok, ttype, att) in enumerate(zip(ids, tokens, ttids, mask), 1):
    if att == 0:
        seg = "[PAD]"
    elif ttype == 0:
        seg = "세그먼트 0  (문장A)"
    else:
        seg = "세그먼트 1  (문장B)"
    print(f"  {pos:4d}   {tid:6d}   {ttype:6d}   {att:5d}   {tok:12}  ← {seg}")


=== 두 문장 동시 입력 ===
  문장A: '이 영화 재미있었어요'
  문장B: '정말 추천합니다'
  입력 형식: [CLS] 문장A [SEP] 문장B [SEP]

    위치       id     type     att   토큰   ← 세그먼트
  ──────────────────────────────────────────────────────────
     1        2        0       1   [CLS]         ← 세그먼트 0  (문장A)
     2      711        0       1   이             ← 세그먼트 0  (문장A)
     3     1979        0       1   영화            ← 세그먼트 0  (문장A)
     4     4823        0       1   재미있었어요        ← 세그먼트 0  (문장A)
     5        3        0       1   [SEP]         ← 세그먼트 0  (문장A)
     6     1988        1       1   정말            ← 세그먼트 1  (문장B)
     7     3972        1       1   추천합니다         ← 세그먼트 1  (문장B)
     8        3        1       1   [SEP]         ← 세그먼트 1  (문장B)
     9        0        0       0   [PAD]         ← [PAD]
    10        0        0       0   [PAD]         ← [PAD]
    11        0        0       0   [PAD]         ← [PAD]
    12        0        0       0   [PAD]         ← [PAD]
    13        0        0       0   [PAD]         ←

---
## 5단계: BPE vs WordPiece 직접 비교

### 🔎 비교 관점

같은 NSMC 말뭉치로 학습한 두 토크나이저를 동일한 문장에 적용합니다.

| 비교 항목 | 확인 내용 |
|----------|----------|
| 토큰 표현 방식 | 한국어가 어떻게 저장/표시되는가 |
| 어절 경계 표시 | `▁` (BPE) vs `##` (WordPiece) |
| 토큰 수 | 같은 문장이 몇 개의 토큰으로 분리되는가 |
| 산출물 파일 | `vocab.json`+`merges.txt` vs `vocab.txt` |


In [ ]:
# BPE vs WordPiece 비교 준비
# ── decode_bbpe_token 재정의 (Part 1 런타임 종료 대비) ────────
def bytes_to_unicode():
    bs = (list(range(ord("!"), ord("~") + 1))
          + list(range(ord("¡"), ord("¬") + 1))
          + list(range(ord("®"), ord("ÿ") + 1)))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}

def decode_bbpe_token(token: str) -> str:
    """BBPE 내부 표현 토큰을 한국어로 변환. 불완전한 UTF-8은 hex로 표시."""
    try:
        byte_list = [byte_decoder[c] for c in token]
    except KeyError:
        return token
    decoded = bytes(byte_list).decode("utf-8", errors="replace")
    if '\ufffd' in decoded:
        return '[' + ' '.join(f'{b:02x}' for b in byte_list) + ']'
    return decoded.replace(' ', '▁')

# ── BPE 토크나이저 로드 ──────────────────────────────────────
from transformers import GPT2Tokenizer

if os.path.exists(BBPE_DIR) and os.listdir(BBPE_DIR):
    tokenizer_gpt = GPT2Tokenizer.from_pretrained(BBPE_DIR)
    tokenizer_gpt.pad_token = "[PAD]"
    print(f"✅ BPE(GPT)  토크나이저 로드 완료  (어휘 집합: {len(tokenizer_gpt):,}개)")
else:
    print("⚠️  BPE 토크나이저가 없습니다. Part 1 노트북을 먼저 실행하세요.")
    tokenizer_gpt = None

print(f"✅ WP(BERT)  토크나이저 확인        (어휘 집합: {len(tokenizer_bert):,}개)")


✅ BPE(GPT)  토크나이저 로드 완료  (어휘 집합: 10,001개)
✅ WP(BERT)  토크나이저 확인        (어휘 집합: 10,000개)


---

> 📌 **Explain 포인트 ⑥** : 아래 비교 결과를 설명하세요.
> - BPE의 `▁` 기호와 WordPiece의 `##` 기호가 각각 무엇을 표시하는가?
> - 같은 문장에서 토큰 수가 다른 이유는 무엇인가?
> - BPE의 `[20 ec]` 같은 hex 표시는 언제 나타나는가?


In [ ]:
# BPE vs WordPiece 토큰화 직접 비교
compare_sentences = [
    "아 더빙.. 진짜 짜증나네요 목소리",
    "흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나",
    "별루 였다..",
    "딥러닝을 공부하는 것은 매우 흥미롭습니다",
    "인공지능이 세상을 바꾸고 있습니다",
]

print("=" * 75)
print("              BPE(GPT) vs WordPiece(BERT) 토큰화 비교")
print("=" * 75)

for i, sent in enumerate(compare_sentences, 1):
    print(f"\n[문장{i}] {sent}")
    print("─" * 65)

    if tokenizer_gpt:
        bpe_tokens   = tokenizer_gpt.tokenize(sent)
        bpe_readable = [decode_bbpe_token(t) for t in bpe_tokens]
        print(f"  BPE   ({len(bpe_tokens):2d}토큰) : {bpe_readable}")
    else:
        print("  BPE   : (Part 1 먼저 실행 필요)")

    wp_tokens  = tokenizer_bert.tokenize(sent)
    wp_display = [f"[{t}]" if t.startswith("##") else t for t in wp_tokens]
    print(f"  WP    ({len(wp_tokens):2d}토큰) : {wp_display}")


              BPE(GPT) vs WordPiece(BERT) 토큰화 비교

[문장1] 아 더빙.. 진짜 짜증나네요 목소리
─────────────────────────────────────────────────────────────────
  BPE   ( 7토큰) : ['아', '▁더빙', '..', '▁진짜', '▁짜증나', '네요', '▁목소리']
  WP    ( 8토큰) : ['아', '더빙', '.', '.', '진짜', '짜증나', '[##네요]', '목소리']

[문장2] 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
─────────────────────────────────────────────────────────────────
  BPE   (15토큰) : ['흠', '...', '포스터', '보고', '▁초딩', '영화', '줄', '....', '오버', '연기', '조차', '▁가볍', '지', '▁않', '구나']
  WP    (20토큰) : ['흠', '.', '.', '.', '포스터', '[##보고]', '초딩', '[##영화]', '[##줄]', '.', '.', '.', '.', '오버', '[##연기]', '[##조차]', '가볍', '[##지]', '않', '[##구나]']

[문장3] 별루 였다..
─────────────────────────────────────────────────────────────────
  BPE   ( 4토큰) : ['별루', '[20 ec 98]', '[80 eb 8b a4]', '..']
  WP    ( 4토큰) : ['별루', '였다', '.', '.']

[문장4] 딥러닝을 공부하는 것은 매우 흥미롭습니다
─────────────────────────────────────────────────────────────────
  BPE   (11토큰) : ['[eb 94]', '[a5]', '러', '닝', '을', '▁공부', '하는', '▁것은', 

In [ ]:
# 정량 비교 — 테스트 데이터 샘플 500문장 통계
print("=== 테스트 데이터 샘플 500문장 정량 비교 ===")
print("(잠시 기다려 주세요...)")

bpe_lengths, wp_lengths = [], []

with open(TEST_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 500: break
        line = line.rstrip()
        if not line: continue
        if tokenizer_gpt:
            bpe_lengths.append(len(tokenizer_gpt.tokenize(line)))
        wp_lengths.append(len(tokenizer_bert.tokenize(line)))

print()
print(f"  {'항목':20}   {'BPE(GPT)':>12}   {'WordPiece(BERT)':>15}")
print("  " + "─" * 55)

if bpe_lengths:
    avg_bpe = sum(bpe_lengths)/len(bpe_lengths)
    med_bpe = sorted(bpe_lengths)[len(bpe_lengths)//2]
    avg_wp  = sum(wp_lengths)/len(wp_lengths)
    med_wp  = sorted(wp_lengths)[len(wp_lengths)//2]
    print(f"  {'평균 토큰 수':20}   {avg_bpe:>12.1f}   {avg_wp:>15.1f}")
    print(f"  {'중앙 토큰 수':20}   {med_bpe:>12d}   {med_wp:>15d}")
    print(f"  {'최대 토큰 수':20}   {max(bpe_lengths):>12d}   {max(wp_lengths):>15d}")
else:
    avg_wp = sum(wp_lengths)/len(wp_lengths)
    print(f"  {'평균 토큰 수(WP)':20}   {'─':>12}   {avg_wp:>15.1f}")

print()
print("=== BPE vs WordPiece 알고리즘 최종 비교 ===")
rows = [
    ("병합 기준",      "빈도(count) 최대화",          "우도(likelihood) 최대화"),
    ("시작 단위",      "UTF-8 바이트",                 "유니코드 문자"),
    ("한국어 저장",    "특수 유니코드로 인코딩",        "그대로 저장"),
    ("어절 경계 표시", "▁ (Ġ 바이트)",                "## prefix"),
    ("산출물 파일",    "vocab.json + merges.txt",     "vocab.txt"),
    ("OOV 처리",      "불가능 (바이트 조합으로 표현)",  "[UNK] 처리"),
    ("특수 토큰",      "[PAD]",                       "[PAD][UNK][CLS][SEP][MASK]"),
    ("추가 입력값",    "input_ids, attention_mask",   "+ token_type_ids"),
    ("대표 모델",      "GPT-2, GPT-3, RoBERTa",      "BERT, ELECTRA, KoBERT"),
]
print(f"\n  {'항목':16}  {'BPE':30}  {'WordPiece'}")
print("  " + "─" * 80)
for item, bpe_val, wp_val in rows:
    print(f"  {item:16}  {bpe_val:30}  {wp_val}")


=== 테스트 데이터 샘플 500문장 정량 비교 ===
(잠시 기다려 주세요...)

  항목                         BPE(GPT)   WordPiece(BERT)
  ───────────────────────────────────────────────────────
  평균 토큰 수                        16.2              16.0
  중앙 토큰 수                          12                12
  최대 토큰 수                          75                70

=== BPE vs WordPiece 알고리즘 최종 비교 ===

  항목                BPE                             WordPiece
  ────────────────────────────────────────────────────────────────────────────────
  병합 기준             빈도(count) 최대화                   우도(likelihood) 최대화
  시작 단위             UTF-8 바이트                       유니코드 문자
  한국어 저장            특수 유니코드로 인코딩                    그대로 저장
  어절 경계 표시          ▁ (Ġ 바이트)                       ## prefix
  산출물 파일            vocab.json + merges.txt         vocab.txt
  OOV 처리            불가능 (바이트 조합으로 표현)               [UNK] 처리
  특수 토큰             [PAD]                           [PAD][UNK][CLS][SEP][MASK]
  추가 입력값            input_ids, at

---
#Remix A — ## prefix 비율 심층 분석
###vocab_size = (1조:2000 / 2조:10000 / 4조:30000)

###실습목적:vocab_size에 따른 ## prefix 토큰 비율 변화 비교


##<실행 방법 및 설정값>
####1. 실험 설정 및 토크나이저 재학습
설정값 변경


- VOCAB_SIZE: 어휘 집합의 최대 크기를 설정합니다.
- tok.train(): 설정된 크기에 맞춰 BertWordPieceTokenizer를 다시 학습시킵니다. 빈도수가 높은 문자 쌍부터 우선적으로 병합하여 새로운 서브워드(Subword)를 생성합니다.
.


####2. vocab 내 ## 비율 계산


*   vocab.txt에서 ##으로 시작하는 항목의 수와 전체 어휘 대비 비율을 계산합니다.
*   BPE에는 없는 WordPiece 고유의 분석 지표입니다.



####3. 문장 토큰화 및 ## 토큰 비율 비교


*   BertTokenizer.tokenize(): 테스트 문장을 토큰화하여 실제 ## 토큰이 얼마나 등장하는지 확인합니다.



####4. 결과 분석 (출력 항목)


*   vocab 내 ## 비율: 전체 어휘 중 ## 서브워드가 차지하는 비율을 확인합니다.
*   토큰 내 ## 비율: 실제 문장 토큰화 시 ## 토큰이 등장하는 비율을 확인합니다.

*   문장별 비교: 가장 작은 vocab_size vs 가장 큰 vocab_size로 동일 문장이 어떻게 달리 분절되는지 직접 확인합니다.



In [ ]:
# [Remix A] ## prefix 비율 심층 분석 (단일 vocab_size 버전)
import os, tempfile
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizer

# ── 실험 설정 ────────────────────────────────────────────
VOCAB_SIZE = 5000   # 👈 1조:2000 / 2조:10000 / 4조:30000

test_sentences = [
    "오늘 날씨가 정말 좋네요",
    "이 영화는 정말 재미없었어요",
    "딥러닝을 공부하는 것은 흥미롭습니다",
    "한국어 자연어처리는 어렵지만 보람있습니다",
    "트랜스포머 모델은 어텐션 메커니즘을 사용합니다",
]
# ─────────────────────────────────────────────────────────

# 1. WordPiece 토크나이저 학습
tmp_dir = tempfile.mkdtemp()
tok = BertWordPieceTokenizer(lowercase=False, strip_accents=False)

tok.train(
    files=[TRAIN_PATH, TEST_PATH],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    wordpieces_prefix="##",
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
)

tok.save_model(tmp_dir)

# 2. vocab.txt에서 ## 비율 계산
with open(os.path.join(tmp_dir, "vocab.txt"), 'r', encoding='utf-8') as f:
    lines = f.readlines()

total_vocab = len(lines)
subword_count = sum(1 for l in lines if l.startswith("##"))
subword_ratio = subword_count / total_vocab * 100

# 3. 테스트 문장 토큰화
bert_tok = BertTokenizer.from_pretrained(tmp_dir, do_lower_case=False)

total_tokens = 0
total_subword_tokens = 0

for sent in test_sentences:
    toks = bert_tok.tokenize(sent)
    total_tokens += len(toks)
    total_subword_tokens += sum(1 for t in toks if t.startswith("##"))

token_sub_ratio = total_subword_tokens / total_tokens * 100 if total_tokens > 0 else 0

# 결과 출력
print("=" * 60)
print(f"  vocab_size = {VOCAB_SIZE}")
print("=" * 60)
print(f"전체 vocab 수        : {total_vocab}")
print(f"## 포함 vocab 수     : {subword_count}")
print(f"👉 vocab 내 ## 비율   : {subword_ratio:.2f}%")
print()
print(f"전체 토큰 수         : {total_tokens}")
print(f"## 토큰 수           : {total_subword_tokens}")
print(f"👉 토큰 내 ## 비율    : {token_sub_ratio:.2f}%")
print("=" * 60)

# 문장별 토큰화 결과 출력
print()
print("  문장별 토큰화 결과")
print("=" * 60)
for sent in test_sentences:
    toks = bert_tok.tokenize(sent)
    display = [f"[{t}]" if t.startswith("##") else t for t in toks]
    sub_count = sum(1 for t in toks if t.startswith("##"))
    print(f"  입력 : {sent}")
    print(f"  토큰 : {display}")
    print(f"  토큰 수: {len(toks)}개  |  ## 토큰: {sub_count}개")

  vocab_size = 5000
전체 vocab 수        : 5000
## 포함 vocab 수     : 2027
👉 vocab 내 ## 비율   : 40.54%

전체 토큰 수         : 45
## 토큰 수           : 24
👉 토큰 내 ## 비율    : 53.33%

  문장별 토큰화 결과
  입력 : 오늘 날씨가 정말 좋네요
  토큰 : ['오늘', '날', '[##씨가]', '정말', '좋네요']
  토큰 수: 5개  |  ## 토큰: 1개
  입력 : 이 영화는 정말 재미없었어요
  토큰 : ['이', '영화는', '정말', '재미없', '[##었어요]']
  토큰 수: 5개  |  ## 토큰: 1개
  입력 : 딥러닝을 공부하는 것은 흥미롭습니다
  토큰 : ['[UNK]', '공부', '[##하는]', '것은', '흥미롭', '[##습니다]']
  토큰 수: 6개  |  ## 토큰: 2개
  입력 : 한국어 자연어처리는 어렵지만 보람있습니다
  토큰 : ['한국', '[##어]', '자연', '[##어]', '[##처]', '[##리는]', '어렵', '[##지만]', '보', '[##람]', '[##있]', '[##습니다]']
  토큰 수: 12개  |  ## 토큰: 8개
  입력 : 트랜스포머 모델은 어텐션 메커니즘을 사용합니다
  토큰 : ['트', '[##랜]', '[##스]', '[##포]', '[##머]', '[UNK]', '어', '[##텐]', '[##션]', '메', '[##커]', '[##니]', '[##즘]', '[##을]', '사', '[##용]', '[##합니다]']
  토큰 수: 17개  |  ## 토큰: 12개


#Remix B — `lowercase=True` vs `False` 비교

### 1. 실습 목적: lowercase 옵션(True/False)에 따른 단어 분할 방식 차이를 확인합니다.
### 2. 실험 설정 :
      lowercase= (1조, 2조 : True / 4조 : False)

<실행 방법 및 설정값>
1. 실험 설정 및 토크나이저 재학습
* 한국어+영어 혼합 문장 20개 학습
* 변경 파라미터: `lowercase = True / False`
→ 대소문자 유지 여부가 토큰 분절에 미치는 영향 분석

2. `vocab.txt` 내 대문자 시작 토큰 수
* `vocab.txt`에서 대문자로 시작하는 토큰 개수 계산
* lowercase 옵션에 따른 변화 확인

3. 문장 토큰화 결과 출력
* 동일한 테스트 문장을 입력하여 토큰화 결과 비교
* 출력 항목:
    원문,
    (lowercase=True) 토큰화 결과,
    (lowercase=False) 토큰화 결과
4. 결과 실시간 비교
* lowercase=True / False 설정값 변경에 따른 토큰화 결과를 대조적으로 비교
* 차이가 발생하는 부분을 중심으로 분석


In [ ]:
# [Remix B] ## lowercase 옵션(True/False)에 따른 단어 분할 방식 차이 확인
import os
import tempfile
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizer

# ── 데이터 구성 ────────────────────────────────────
test_sentences_mixed = [
    "AI 기술이 발전하고 있습니다",
    "BERT와 GPT의 차이점이 궁금합니다",
    "Deep Learning은 매우 강력합니다",
    "Python으로 NLP를 배워봅시다",
    "Transformer 모델의 Attention 메커니즘",
    "ChatGPT는 OpenAI가 만든 모델입니다",
    "NSMC 데이터셋으로 감성 분석을 합니다",
    "KoBERT는 한국어 BERT 모델입니다",
    "Word2Vec과 GloVe는 임베딩 방법입니다",
    "RNN에서 LSTM으로 발전했습니다",
    "한국어 tokenization은 어렵습니다",
    "ELECTRA는 BPE를 사용하지 않습니다",
    "GPU로 학습 속도를 높입니다",
    "loss function은 CrossEntropy입니다",
    "batch_size를 늘리면 빠릅니다",
    "Adam optimizer를 사용합니다",
    "dropout으로 과적합을 방지합니다",
    "Accuracy와 F1-Score를 측정합니다",
    "HuggingFace의 tokenizers 라이브러리",
    "PyTorch와 TensorFlow 비교",
]

# ── 실험 설정 ─────────────────────────────
LOWERCASE = False  # 👈 1조,2조: True / 4조: False
# ───────────────────────────────────────

tmp_dir = tempfile.mkdtemp()

tok = BertWordPieceTokenizer(lowercase=LOWERCASE, strip_accents=False)
tok.train(
    files=[TRAIN_PATH, TEST_PATH],
    vocab_size=10000,
    min_frequency=2,
    wordpieces_prefix="##",
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
)

tok.save_model(tmp_dir)

# vocab 분석
with open(os.path.join(tmp_dir, "vocab.txt"), 'r', encoding='utf-8') as f:
    vocab_lines = f.readlines()

uppercase_count = sum(
    1 for l in vocab_lines
    if l.strip()
    and not l.startswith("[")
    and not l.startswith("##")
    and l.strip()[0].isupper()
)

bert_tok = BertTokenizer.from_pretrained(tmp_dir, do_lower_case=LOWERCASE)

# 토큰화 결과 출력
print("=" * 60)
print(f"lowercase = {LOWERCASE}")
print(f"대문자 시작 토큰 수: {uppercase_count:,}개")
print("=" * 60)

print("\n문장별 토큰화 결과")
print("=" * 60)

for sent in test_sentences_mixed:
    toks = bert_tok.tokenize(sent)
    print(f"문장: {sent}")
    print(f"토큰: {toks}")
    print()

# 결과 저장용 딕셔너리
if 'lowercase_settings' not in globals():
    lowercase_settings = {}

# 현재 결과 저장
lowercase_settings[LOWERCASE] = {
    'tok': bert_tok,
    'uppercase_count': uppercase_count
}

lowercase = False
대문자 시작 토큰 수: 43개

문장별 토큰화 결과
문장: AI 기술이 발전하고 있습니다
토큰: ['A', '##I', '기술', '##이', '발전', '##하고', '있습니다']

문장: BERT와 GPT의 차이점이 궁금합니다
토큰: ['B', '##E', '##R', '##T', '##와', 'G', '##P', '##T', '##의', '차이', '##점이', '궁금', '##합니다']

문장: Deep Learning은 매우 강력합니다
토큰: ['D', '##e', '##e', '##p', 'L', '##e', '##ar', '##n', '##ing', '##은', '매우', '강력', '##합니다']

문장: Python으로 NLP를 배워봅시다
토큰: ['P', '##y', '##t', '##h', '##on', '##으로', 'N', '##L', '##P', '##를', '배워', '##봅', '##시다']

문장: Transformer 모델의 Attention 메커니즘
토큰: ['T', '##r', '##an', '##s', '##f', '##or', '##m', '##er', '[UNK]', 'A', '##tt', '##en', '##t', '##i', '##on', '메', '##커', '##니', '##즘']

문장: ChatGPT는 OpenAI가 만든 모델입니다
토큰: ['C', '##h', '##at', '##G', '##P', '##T', '##는', 'O', '##p', '##en', '##A', '##I', '##가', '만든', '[UNK]']

문장: NSMC 데이터셋으로 감성 분석을 합니다
토큰: ['N', '##S', '##M', '##C', '[UNK]', '감성', '분', '##석', '##을', '합니다']

문장: KoBERT는 한국어 BERT 모델입니다
토큰: ['K', '##o', '##B', '##E', '##R', '##T', '##는', '한국', '##어', 'B', '##

In [ ]:
import tempfile
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizer
import os

# ── 테스트 문장 (한국어+영어 혼합)
test_sentences_mixed = [
    "AI 기술이 발전하고 있습니다",
    "BERT와 GPT의 차이점이 궁금합니다",
    "Deep Learning은 매우 강력합니다",
    "Python으로 NLP를 배워봅시다",
    "Transformer 모델의 Attention 메커니즘",
    "ChatGPT는 OpenAI가 만든 모델입니다",
    "NSMC 데이터셋으로 감성 분석을 합니다",
    "KoBERT는 한국어 BERT 모델입니다",
    "Word2Vec과 GloVe는 임베딩 방법입니다",
    "RNN에서 LSTM으로 발전했습니다",
    "한국어 tokenization은 어렵습니다",
    "ELECTRA는 BPE를 사용하지 않습니다",
    "GPU로 학습 속도를 높입니다",
    "loss function은 CrossEntropy입니다",
    "batch_size를 늘리면 빠릅니다",
    "Adam optimizer를 사용합니다",
    "dropout으로 과적합을 방지합니다",
    "Accuracy와 F1-Score를 측정합니다",
    "HuggingFace의 tokenizers 라이브러리",
    "PyTorch와 TensorFlow 비교",
]

# ── 두 설정 모두 학습
lowercase_settings = {}
for lc in [False, True]:
    tmp_dir = tempfile.mkdtemp()
    tok = BertWordPieceTokenizer(lowercase=lc, strip_accents=False)
    tok.train(
        files=[TRAIN_PATH, TEST_PATH],
        vocab_size=10000,
        min_frequency=2,
        wordpieces_prefix="##",
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
    )
    tok.save_model(tmp_dir)

    with open(os.path.join(tmp_dir, "vocab.txt"), 'r', encoding='utf-8') as f:
        vocab_lines = f.readlines()

    uppercase_count = sum(
        1 for l in vocab_lines
        if l.strip() and not l.startswith("[") and not l.startswith("##") and l.strip()[0].isupper()
    )

    bert_tok = BertTokenizer.from_pretrained(tmp_dir, do_lower_case=lc)
    lowercase_settings[lc] = {
        "tok": bert_tok,
        "uppercase_count": uppercase_count,
    }


# ── 토큰화 차이 비교
tok_false = lowercase_settings[False]['tok']
tok_true  = lowercase_settings[True]['tok']

print("=" * 72)
print("  lowercase 설정에 따른 토큰화 결과 비교")
print("=" * 72)

diff_sentences = []

for i, sent in enumerate(test_sentences_mixed, 1):
    t_false = tok_false.tokenize(sent)
    t_true  = tok_true.tokenize(sent)

    # 차이 여부
    if t_false != t_true:
        diff_sentences.append(sent)
        marker = " ← 차이!"
    else:
        marker = ""

    print(f"[{i:02d}] {sent[:33]:35}")
    print(f"     False: {t_false}")
    print(f"     True : {t_true}{marker}")
    print("-" * 72)

# 총 차이 문장 수
print(f"\n👉 차이가 나는 문장 수: {len(diff_sentences)} / {len(test_sentences_mixed)}\n")

# vocab 대문자 토큰 비교
uc_false = lowercase_settings[False]['uppercase_count']
uc_true  = lowercase_settings[True]['uppercase_count']
print(f"vocab 내 대문자 시작 토큰 수:")
print(f"  lowercase=False : {uc_false:,}개  (대소문자 구분 → 대문자 토큰 존재)")
print(f"  lowercase=True  : {uc_true:,}개  (소문자 변환 → 대문자 토큰 거의 없음)")

  lowercase 설정에 따른 토큰화 결과 비교
[01] AI 기술이 발전하고 있습니다                   
     False: ['A', '##I', '기술', '##이', '발전', '##하고', '있습니다']
     True : ['a', '##i', '[UNK]', '[UNK]', '[UNK]'] ← 차이!
------------------------------------------------------------------------
[02] BERT와 GPT의 차이점이 궁금합니다              
     False: ['B', '##E', '##R', '##T', '##와', 'G', '##P', '##T', '##의', '차이', '##점이', '궁금', '##합니다']
     True : ['[UNK]', '[UNK]', '[UNK]', '[UNK]'] ← 차이!
------------------------------------------------------------------------
[03] Deep Learning은 매우 강력합니다            
     False: ['D', '##e', '##e', '##p', 'L', '##e', '##ar', '##n', '##ing', '##은', '매우', '강력', '##합니다']
     True : ['d', '##e', '##e', '##p', '[UNK]', '[UNK]', '[UNK]'] ← 차이!
------------------------------------------------------------------------
[04] Python으로 NLP를 배워봅시다                
     False: ['P', '##y', '##t', '##h', '##on', '##으로', 'N', '##L', '##P', '##를', '배워', '##봅', '##시다']
     True : ['[UNK]', '[UNK]', '[UNK]

---

#Remix C — 두 문장 입력 & NSP 탐구
###실습 목적:.BERT는 NSP 학습을 위해 [CLS] A [SEP] B [SEP] 구조로 두 문장을 입력받습니다. 이번 실험에서는 NSP 입력 형식을 직접 구현하며, 문장 길이에 따라 token_type_ids 경계와 truncation이 어떻게 달라지는지 탐구합니다.
###max_length= (1조: max_length=16 / 2조: max_length=24 / 4조: max_length=32)


##<실행 방법 및 설정값>
####1. 실험 설정 및 문장 쌍 준비
설정값 변경

- MAX_LENGTH: 입력 시퀀스의 최대 길이를 설정합니다.
- 짧은A+짧은B / 긴A+짧은B / 짧은A+긴B / 동일 길이 / 영어A+한국어B 총 5가지 문장 쌍을 사용합니다.

####2. 두 문장 토큰화

- tokenizer_bert(): 두 문장을 동시에 입력받아 [CLS] 문장A [SEP] 문장B [SEP] 형식으로 변환합니다.
- padding="max_length" + truncation=True: 설정한 MAX_LENGTH에 맞춰 패딩 및 자르기를 적용합니다.

####3. token_type_ids 경계 분석

- token_type_ids: 0은 문장A(세그먼트 0), 1은 문장B(세그먼트 1)를 나타냅니다.
- 0→1로 바뀌는 경계 위치를 탐지하여 두 문장의 분리 지점을 시각적으로 확인합니다.

####4. 결과 분석 (출력 항목)

- 위치 / token / type / att: 각 토큰의 위치, 토큰 값, 세그먼트 구분, 어텐션 마스크를 확인합니다.
- 0→1 경계 위치: 문장 길이에 따라 경계가 어떻게 달라지는지 확인합니다.
- truncation 발생 여부: max_length 초과 시 문장B의 뒷부분부터 잘리는지 직접 확인합니다.

In [ ]:
# =========================
# 0. 라이브러리
# =========================
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)


# =========================
# 1. 문장 쌍
# =========================
sentence_pairs = [
    ("오늘 날씨 좋다", "산책 가고 싶다", "짧은A + 짧은B"),

    ("트랜스포머 모델은 어텐션 메커니즘을 기반으로 문맥 정보를 효과적으로 반영하며 다양한 자연어처리 작업에서 뛰어난 성능을 보입니다",
     "맞아요", "긴A + 짧은B"),

    ("네",
     "WordPiece 토크나이저는 단어를 서브워드 단위로 분해하여 희귀 단어 문제를 해결하며 다양한 언어에서 효율적인 토큰화를 가능하게 합니다",
     "짧은A + 긴B"),

    ("BERT는 문맥을 이해하는 모델입니다",
     "GPT는 텍스트를 생성하는 모델입니다",
     "동일 길이"),

    ("Natural language processing is an important field in artificial intelligence",
     "자연어처리는 인공지능에서 매우 중요한 분야입니다",
     "영어A + 한국어B"),
]


# =========================
# 2. 설정
# =========================
MAX_LENGTH = 16   # 🔥 (1조: max_length=16 / 2조: max_length=24 / 4조: max_length=32)
print("=" * 75)
print(f"  두 문장 입력 실험  (max_length={MAX_LENGTH})")
print("=" * 75)


# =========================
# 3. 실험 시작
# =========================
for sent_a, sent_b, desc in sentence_pairs:

    pair_input = tokenizer_bert(
        sent_a, sent_b,
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    ids    = pair_input['input_ids']
    ttids  = pair_input['token_type_ids']
    mask   = pair_input['attention_mask']
    tokens = tokenizer_bert.convert_ids_to_tokens(ids)

    # ── A → B 경계 찾기 ──
    boundary = -1
    for i in range(1, len(ttids)):
        if ttids[i-1] == 0 and ttids[i] == 1:
            boundary = i
            break

    # ── truncation 여부 ──
    actual_a = tokenizer_bert.tokenize(sent_a)
    actual_b = tokenizer_bert.tokenize(sent_b)
    original_len = len(actual_a) + len(actual_b) + 3
    truncated = original_len > MAX_LENGTH

    # =========================
    # DataFrame 생성
    # =========================
    rows = []

    for pos, (tid, tok, ttype, att) in enumerate(zip(ids, tokens, ttids, mask), 1):

        if att == 0:
            seg = "PAD"
        elif ttype == 0:
            seg = "A"
        else:
            seg = "B"

        # 특수 토큰 표시
        if tok == "[CLS]":
            special = "CLS"
        elif tok == "[SEP]":
            special = "SEP"
        else:
            special = ""

        # 경계 표시
        boundary_mark = "A→B" if pos == boundary + 1 else ""

        rows.append({
            "위치": pos,
            "토큰": tok,
            "세그먼트": seg,
            "type_id": ttype,
            "attention": att,
            "특수토큰": special,
            "경계": boundary_mark
        })

    df = pd.DataFrame(rows)

    # =========================
    # 출력
    # =========================
    print(f"\n[{desc}]")
    print(f"문장A 토큰 수: {len(actual_a)} / 문장B 토큰 수: {len(actual_b)}")

    if truncated:
        print(f"⚠️ truncation 발생 (원래 길이 {original_len})")

    if boundary >= 0:
        print(f"👉 A→B 경계 위치: {boundary + 1}")

    display(df)

    # =========================
    # 🔥 truncation 분석 (핵심 추가)
    # =========================
    real_tokens = [tok for tok, m in zip(tokens, mask) if m == 1]

    sep_indices = [i for i, tok in enumerate(real_tokens) if tok == "[SEP]"]

    if len(sep_indices) >= 2:
        a_tokens = real_tokens[1:sep_indices[0]]
        b_tokens = real_tokens[sep_indices[0]+1:sep_indices[1]]
    else:
        a_tokens, b_tokens = [], []

    print(f"👉 실제 B 토큰 길이: {len(actual_b)}")
    print(f"👉 입력된 B 토큰 길이: {len(b_tokens)}")

    if truncated:
        if len(b_tokens) < len(actual_b):
            print("👉 결과: B 문장의 '뒤쪽'이 잘림 (기본 truncation 방식)")
        else:
            print("👉 결과: B 문장은 유지됨")


# =========================
# 4. 해석
# =========================
print("\n\n📌 해석 포인트:")
print("  - [CLS]는 입력 전체의 시작 토큰이다")
print("  - 첫 번째 [SEP]까지가 세그먼트 A (문장 A)")
print("  - 두 번째 [SEP]까지가 세그먼트 B (문장 B)")
print("  - token_type_ids는 각 토큰이 어느 문장에 속하는지 나타낸다")
print("  - attention_mask는 실제 토큰(1)과 padding(0)을 구분한다")
print("  - max_length 초과 시 truncation이 발생한다")
print("  - 기본 설정에서는 긴 문장에서 뒤쪽 토큰이 우선적으로 제거된다")


  두 문장 입력 실험  (max_length=16)

[짧은A + 짧은B]
문장A 토큰 수: 4 / 문장B 토큰 수: 4
👉 A→B 경계 위치: 7


,위치,토큰,세그먼트,type_id,attention,특수토큰,경계
0,1,[CLS],A,0,1,CLS,
1,2,오늘,A,0,1,,
2,3,날,A,0,1,,
3,4,##씨,A,0,1,,
4,5,좋다,A,0,1,,
5,6,[SEP],A,0,1,SEP,
6,7,산,B,1,1,,A→B
7,8,##책,B,1,1,,
8,9,가고,B,1,1,,
9,10,싶다,B,1,1,,


👉 실제 B 토큰 길이: 4
👉 입력된 B 토큰 길이: 4

[긴A + 짧은B]
문장A 토큰 수: 35 / 문장B 토큰 수: 2
⚠️ truncation 발생 (원래 길이 40)
👉 A→B 경계 위치: 14


,위치,토큰,세그먼트,type_id,attention,특수토큰,경계
0,1,[CLS],A,0,1,CLS,
1,2,트랜스포머,A,0,1,,
2,3,[UNK],A,0,1,,
3,4,어,A,0,1,,
4,5,##텐,A,0,1,,
5,6,##션,A,0,1,,
6,7,메,A,0,1,,
7,8,##커,A,0,1,,
8,9,##니,A,0,1,,
9,10,##즘,A,0,1,,


👉 실제 B 토큰 길이: 2
👉 입력된 B 토큰 길이: 2
👉 결과: B 문장은 유지됨

[짧은A + 긴B]
문장A 토큰 수: 1 / 문장B 토큰 수: 42
⚠️ truncation 발생 (원래 길이 46)
👉 A→B 경계 위치: 4


...

👉 실제 B 토큰 길이: 42
👉 입력된 B 토큰 길이: 12
👉 결과: B 문장의 '뒤쪽'이 잘림 (기본 truncation 방식)

[동일 길이]
문장A 토큰 수: 11 / 문장B 토큰 수: 9
⚠️ truncation 발생 (원래 길이 23)
👉 A→B 경계 위치: 10


,위치,토큰,세그먼트,type_id,attention,특수토큰,경계
0,1,[CLS],A,0,1,CLS,
1,2,B,A,0,1,,
2,3,##E,A,0,1,,
3,4,##R,A,0,1,,
4,5,##T,A,0,1,,
5,6,##는,A,0,1,,
6,7,문,A,0,1,,
7,8,##맥,A,0,1,,
8,9,[SEP],A,0,1,SEP,
9,10,G,B,1,1,,A→B


👉 실제 B 토큰 길이: 9
👉 입력된 B 토큰 길이: 6
👉 결과: B 문장의 '뒤쪽'이 잘림 (기본 truncation 방식)

[영어A + 한국어B]
문장A 토큰 수: 50 / 문장B 토큰 수: 14
⚠️ truncation 발생 (원래 길이 67)
👉 A→B 경계 위치: 10


,위치,토큰,세그먼트,type_id,attention,특수토큰,경계
0,1,[CLS],A,0,1,CLS,
1,2,N,A,0,1,,
2,3,##at,A,0,1,,
3,4,##u,A,0,1,,
4,5,##r,A,0,1,,
5,6,##al,A,0,1,,
6,7,l,A,0,1,,
7,8,##an,A,0,1,,
8,9,[SEP],A,0,1,SEP,
9,10,자연,B,1,1,,A→B


👉 실제 B 토큰 길이: 14
👉 입력된 B 토큰 길이: 6
👉 결과: B 문장의 '뒤쪽'이 잘림 (기본 truncation 방식)


📌 해석 포인트:
  - [CLS]는 입력 전체의 시작 토큰이다
  - 첫 번째 [SEP]까지가 세그먼트 A (문장 A)
  - 두 번째 [SEP]까지가 세그먼트 B (문장 B)
  - token_type_ids는 각 토큰이 어느 문장에 속하는지 나타낸다
  - attention_mask는 실제 토큰(1)과 padding(0)을 구분한다
  - max_length 초과 시 truncation이 발생한다
  - 기본 설정에서는 긴 문장에서 뒤쪽 토큰이 우선적으로 제거된다


---

### Remix D — `[MASK]` 토큰 삽입 실험 (MLM 시뮬레이션)

**BPE 팀과 다른 점**: BPE에는 `[MASK]` 토큰이 없습니다. BERT의 핵심 학습 방식 실험입니다.

**실험 아이디어**

BERT의 학습 방식인 MLM(Masked Language Model)을 직접 시뮬레이션합니다.

실제 BERT 학습에서는 전체 토큰의 약 **15%를 무작위로** `[MASK]`로 대체합니다.  
(이 중 80%는 [MASK]로, 10%는 랜덤 토큰으로, 10%는 원래 토큰으로 대체)

이 실험에서는 위 규칙을 직접 구현하여 BERT가 학습 중 어떤 형태의 입력을 받는지 확인합니다.

**실험 설계 예시**
1. 예시 문장을 토큰화하여 `input_ids`를 구합니다.
2. 전체 토큰의 15%를 무작위로 선택합니다.
3. 선택된 토큰 중 80%는 `[MASK]`로, 10%는 랜덤 토큰으로, 10%는 원래 그대로 둡니다.
4. 마스킹된 결과를 출력하고, 어떤 토큰이 가려졌는지 확인하세요.


In [ ]:
# [Remix D] [MASK] 토큰 삽입 실험 (MLM 시뮬레이션)
import random

MASK_PROB    = 0.15   # 👈 1조:0.15 / 2조:0.3 / 4조:0.5
MASK_TOKEN   = 0.80
RANDOM_TOKEN = 0.10


MASK_ID    = tokenizer_bert.mask_token_id
VOCAB_SIZE = len(tokenizer_bert)

demo_sentence = "딥러닝을 배우는 것은 어렵지만 매우 흥미롭습니다"

# 1. 문장 인코딩
encoded = tokenizer_bert.encode(demo_sentence, add_special_tokens=True)
original_ids = encoded[:]           # 원본 보존 (비교용)
masked_ids = encoded[:]             # 실제로 마스킹할 복사본

# special token id
cls_id = tokenizer_bert.cls_token_id
sep_id = tokenizer_bert.sep_token_id

# 2. special token 제외한 위치 찾기
candidate_positions = [
    i for i, token_id in enumerate(original_ids)
    if token_id not in [cls_id, sep_id]
]

# 마스킹할 개수 계산 (최소 1개)
num_to_mask = max(1, int(len(candidate_positions) * MASK_PROB))

# 무작위 위치 선택
masked_positions = random.sample(candidate_positions, num_to_mask)

# 어떤 방식으로 바뀌었는지 기록
masking_log = []

# 3. 선택된 위치에 규칙 적용
for pos in masked_positions:
    prob = random.random()
    original_token_id = masked_ids[pos]

    if prob < MASK_TOKEN:
        # 80% -> [MASK]
        masked_ids[pos] = MASK_ID
        change_type = "[MASK]"
    elif prob < MASK_TOKEN + RANDOM_TOKEN:
        # 10% -> 랜덤 토큰
        random_id = random.randint(5, VOCAB_SIZE - 1)
        masked_ids[pos] = random_id
        change_type = "RANDOM"
    else:
        # 10% -> 원래 유지
        change_type = "KEEP"

    masking_log.append({
        "position": pos,
        "before": tokenizer_bert.convert_ids_to_tokens([original_token_id])[0],
        "after": tokenizer_bert.convert_ids_to_tokens([masked_ids[pos]])[0],
        "type": change_type
    })

# 4. 토큰 문자열 변환
original_tokens = tokenizer_bert.convert_ids_to_tokens(original_ids)
masked_tokens = tokenizer_bert.convert_ids_to_tokens(masked_ids)

print("원본 토큰 시퀀스:")
print(original_tokens)

print("\n마스킹 후 토큰 시퀀스:")
print(masked_tokens)

print("\n마스킹 상세 정보:")
for log in masking_log:
    print(
        f"위치 {log['position']}: "
        f"{log['before']} -> {log['after']} ({log['type']})"
    )

원본 토큰 시퀀스:
['[CLS]', '[UNK]', '배우는', '것은', '어렵', '##지만', '매우', '흥미롭', '##습니다', '[SEP]']

마스킹 후 토큰 시퀀스:
['[CLS]', '[UNK]', '배우는', '[MASK]', '어렵', '##지만', '매우', '흥미롭', '##습니다', '[SEP]']

마스킹 상세 정보:
위치 3: 것은 -> [MASK] ([MASK])


---

### Remix E — `[UNK]` 발생 패턴 분석

**BPE 팀과 다른 점**: BPE는 바이트 조합으로 OOV를 표현하므로 `[UNK]`가 발생하지 않습니다.  
WordPiece의 `[UNK]` 처리 방식은 BPE와의 핵심 차이점입니다.

**실험 아이디어**

WordPiece에서 `[UNK]`가 언제, 얼마나 발생하는지 분석합니다.

- WordPiece는 어절 전체를 처리할 서브워드 후보가 하나도 없을 때 어절 전체를 `[UNK]`로 처리합니다.
- 어휘 집합이 작을수록 `[UNK]`가 더 많이 발생합니다.
- 신조어, 외래어, 오타 등 특이한 텍스트에서 `[UNK]`가 많이 발생합니다.

**실험 설계 예시**
1. 테스트 데이터 1,000문장을 토큰화하여 `[UNK]` 발생 횟수를 계산합니다.
2. `[UNK]`가 많이 발생하는 토큰의 공통 특징은 무엇인가요? (신조어? 영어? 특수문자?)
3. `[UNK]`가 자주 발생하는 문장 TOP 10을 출력합니다.


In [ ]:
import re

UNK_ID = tokenizer_bert.unk_token_id  # [UNK]의 인덱스 = 1

# ──────────────────────────────────────────────────────────────
# STEP 1: 테스트 데이터 1000문장 로드
# ──────────────────────────────────────────────────────────────
sentences = []
with open(TEST_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 1000: break
        line = line.rstrip()
        if line:
            sentences.append(line)

print(f"✅ 테스트 문장 수: {len(sentences)}개\n")

# ──────────────────────────────────────────────────────────────
# STEP 2: 전체 UNK 비율 계산 및 전체 1000문장 어절 수집
# ──────────────────────────────────────────────────────────────
total_tokens = 0
total_unks   = 0
unk_words    = []  # 1000개 문장 전체에서 수집할 UNK 어절 목록

for sent in sentences:
    # 1. 문장 전체 토큰화 (비율 계산용)
    ids = tokenizer_bert.encode(sent, add_special_tokens=False)
    total_tokens += len(ids)
    total_unks   += ids.count(UNK_ID)

    # 2. 어절 단위 분석 (1000문장 전체 대상 UNK 유발 어절 수집)
    for word in sent.split():
        tokens = tokenizer_bert.tokenize(word)
        if '[UNK]' in tokens:
            unk_words.append(word)

print("=" * 60)
print(f"  [vocab_size=10000] 전체 UNK 분석 결과")
print("=" * 60)
print(f"  총 토큰 수       : {total_tokens:,}개")
print(f"  총 [UNK] 수      : {total_unks:,}개")
print(f"  [UNK] 비율       : {total_unks / total_tokens * 100:.4f}%\n")



# ──────────────────────────────────────────────────────────────
# STEP 3: [UNK] 가장 많이 발생한 문장 TOP 10
# ──────────────────────────────────────────────────────────────
print("=" * 60)
print("  [UNK] 최다 발생 문장 TOP 10")
print("=" * 60)

sentence_unk_counts = []

for sent in sentences:
    ids = tokenizer_bert.encode(sent, add_special_tokens=False)
    unk_count = ids.count(UNK_ID)
    if unk_count > 0:
        sentence_unk_counts.append((sent, unk_count))

# UNK 많은 순 정렬
top10 = sorted(sentence_unk_counts, key=lambda x: x[1], reverse=True)[:10]

for rank, (sent, count) in enumerate(top10, 1):
    print(f"  [{rank:2d}위] UNK {count}개")
    print(f"       {sent[:60]}{'...' if len(sent) > 60 else ''}\n")

# ──────────────────────────────────────────────────────────────
# STEP 4: [UNK] 발생 어절 공통 특징 분석 (전체 1000문장 기준)
# ──────────────────────────────────────────────────────────────
print("=" * 60)
print("  [UNK] 발생 어절 공통 특징 분석 (전체 1000문장 기준)")
print("=" * 60)

# 분석의 정확도를 위해 중복으로 발생한 UNK 어절 제거 (예: ㅋㅋ, ㅠㅠ 중복 카운트 방지)
unique_unk_words = list(set(unk_words))

print(f"  총 [UNK] 발생 어절 수 (중복 포함) : {len(unk_words):,}개")
print(f"  고유 [UNK] 어절 수 (중복 제거)    : {len(unique_unk_words):,}개\n")

# 특징별 분류 (고유 어절 기준)
english  = [w for w in unique_unk_words if re.search(r'[a-zA-Z]', w)]
special  = [w for w in unique_unk_words if re.search(r'[^\w가-힣a-zA-Z0-9\s]', w)]
numbers  = [w for w in unique_unk_words if re.search(r'\d', w)]
korean   = [w for w in unique_unk_words if re.search(r'[가-힣]', w) and not re.search(r'[a-zA-Z\d]', w)]

print(f"  영문 포함 어절   : {len(english):,}개  {english[:5]}")
print(f"  특수문자 포함    : {len(special):,}개  {special[:5]}")
print(f"  숫자 포함 어절   : {len(numbers):,}개  {numbers[:5]}")
print(f"  한국어만 어절    : {len(korean):,}개  {korean[:5]}\n")

# ──────────────────────────────────────────────────────────────
# STEP 5: vocab_size별 [UNK] 발생 비교
# ──────────────────────────────────────────────────────────────
from tokenizers import BertWordPieceTokenizer

print("=" * 60)
print("  [STEP 5] vocab_size별 [UNK] 발생 비교")
print("=" * 60)



tokenizer_bert_2000 = BertWordPieceTokenizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=False,
    lowercase=False,
    unk_token="[UNK]"
)

#[STEP 5] vocab_size=2000,10000 비교
tokenizer_bert_2000.train(
    files=[TRAIN_PATH],  # <--- TEST_PATH에서 TRAIN_PATH로 변경됨!
    vocab_size=2000,
    limit_alphabet=1000,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
    wordpieces_prefix="##"
)


UNK_ID_2000 = tokenizer_bert_2000.token_to_id("[UNK]")
total_tokens_2000 = 0
total_unks_2000   = 0

# sentences는 앞서 TEST_PATH에서 뽑아둔 1,000문장입니다.
for sent in sentences:
    # ids 리스트를 뽑아옵니다
    encoded = tokenizer_bert_2000.encode(sent)
    ids = encoded.ids

    total_tokens_2000 += len(ids)
    total_unks_2000   += ids.count(UNK_ID_2000)

# 4. 결과 전격 비교 출력
print(f"  ✅ 결과 비교")
print(f"  - [사전 크기 10,000] 일 때 [UNK] 발생: {total_unks:>5,}개 ({total_unks / total_tokens * 100:.2f}%)")
print(f"  - [사전 크기  2,000] 일 때 [UNK] 발생: {total_unks_2000:>5,}개 ({total_unks_2000 / total_tokens_2000 * 100:.2f}%)")
print()



✅ 테스트 문장 수: 1000개

  [vocab_size=10000] 전체 UNK 분석 결과
  총 토큰 수       : 16,195개
  총 [UNK] 수      : 178개
  [UNK] 비율       : 1.0991%

  [UNK] 최다 발생 문장 TOP 10
  [ 1위] UNK 4개
       쇼미더 머니 ㅡㅡ왜하는건가요 힙합보여줄라고 하는거 아닌가요?롹하고 ㅡㅡ 롹이 힙합은 아니자나요 ㅡㅡ아.. 점...

  [ 2위] UNK 4개
       예전이나 지금이나 별로인영화 신선한건 좋은데음향이 엉망 볼륨조절에 실패 송학림이 얘기할땐 볼륨 최대로 맞춰야...

  [ 3위] UNK 3개
       1점고 아깝다. 개막장 영화의 원조라고나 할까.아내와 사별한 지 얼마나 지났다고 딴 여자들 만나고 다니다가 ...

  [ 4위] UNK 3개
       캡틴 듀브아 나올 때마다 너무 웃겼다. 줄리안이랑 곰의 러브스토리도 재밌었고 무엇보다 서커스씬은 노래 fir...

  [ 5위] UNK 3개
       누구나 한 번쯤 들어봤을 ‘그레이스 켈리’에 대한 이 영화는 왕비/여배우로서 주인공이 느끼는 갈등과 모나코와...

  [ 6위] UNK 2개
       셰익스피어는 셰익스피어고 이영화는 이영화.

  [ 7위] UNK 2개
       개성있는 몬스터들과 귀여운 여자아이가 만나 뿜어내는 매력에 퐁당 빠졌다

  [ 8위] UNK 2개
       마트라는 폐쇄된 공간속에서 그안에 갇힌 사람들의 심리변화가 인상적이네요. 결말이 인상적입니다.

  [ 9위] UNK 2개
       아직까지 잊지 못하는 영화. 제목부터 마음에 잔잔하게 와 닿는다. 영화 장면중에 소낙비를 피해놀라 달려 가는...

  [10위] UNK 2개
       몰입도 제로 뱀파이어가 피 조금 흘려서 자살하는 꼴에 늑대인간과 뱀파이어라는 초자연적인 존재의 총격전은 왜 ...

  [UNK] 발생 어절 공통 특징 분석 (전체 1000문장 기준)
  총 [UNK] 발생 

------------------------------------------------------------------

#🎬 Demo 구역 : BPE ↔ WordPiece 나란히 비교
###실험 목적: 같은 문장을 BPE와 WordPiece로 각각 토큰화하여 어절 분리 방식의 차이를 나란히 비교해보자!


##<실행 방법 및 설정값>

###1. BPE 토크나이저 로드
- Part 1 노트북에서 학습한 BPE 토크나이저를 불러옵니다.

- decode_bbpe_token(): BBPE 내부 표현 토큰을 한국어로 변환하여 가독성을 높입니다. ▁로 어절 경계를 시각적으로 표시합니다.

###2. 데모 문장 설정
- 추천: 영화 데이터에서 자주 쓰이는 단어와 신조어·전문용어가 섞인 문장을 다양하게 입력해서 두 방식의 차이를 비교해보세요!

###3. [Demo 1] BPE vs WordPiece 전체 토큰 수 비교

- BPE 결과: 한국어로 복원된 토큰 리스트와 ▁로 어절 경계를 표시합니다.
- WordPiece 결과: [##토큰] 형식으로 서브워드를 강조하여 출력합니다.

###4. [Demo 2] 어절 경계 표현 비교 (▁ vs ##)

- BPE는 ▁로 어절의 시작을, WordPiece는 ##으로 이전 토큰과 이어짐을 표시합니다.
- 같은 문장에서 두 방식이 어절 경계를 어떻게 다르게 표현하는지 확인합니다.

###5. [Demo 3] 어절별 토큰 수 비교

- 입력 문장의 각 어절을 BPE / WordPiece로 각각 토큰화하여 토큰 수를 나란히 비교합니다.
- 두 방식 간 토큰 수 차이가 가장 큰 어절을 자동으로 찾아 강조 표시합니다.

###6. 핵심 해석

- 두 토크나이저의 어절 경계 표현 방식과 분할 기준의 차이를 종합적으로 정리합니다.


##1. BPE 토크나이저 로드
#### Part 1 노트북에서 학습한 BPE 토크나이저를 불러옵니다.
####decode_bbpe_token()은 BBPE 내부 표현 토큰을 한국어로 변환하여 가독성을 높이고, ▁로 어절 경계를 시각적으로 표시합니다.

In [ ]:
# [Demo] BPE ↔ WordPiece 나란히 비교

# ── BPE 토크나이저 로드 (Part 1 노트북 결과 필요) ────────
from tokenizers import ByteLevelBPETokenizer

bbpe_tokenizer = ByteLevelBPETokenizer()
# here

print("🔄 BBPE 토크나이저 학습 중... (약 1~2분 소요)")
bbpe_tokenizer.train(
    files=[TRAIN_PATH, TEST_PATH],
    vocab_size=10000,
    min_frequency=2,
    special_tokens=["[PAD]"],
)

# 저장: vocab.json (어휘 집합) + merges.txt (병합 우선순위)
bbpe_tokenizer.save_model(BBPE_DIR)

print(f"\n✅ BBPE 토크나이저 저장 완료: {BBPE_DIR}")
for fname in sorted(os.listdir(BBPE_DIR)):
    fpath = os.path.join(BBPE_DIR, fname)
    print(f"   📄 {fname}  ({os.path.getsize(fpath):,} bytes)")

🔄 BBPE 토크나이저 학습 중... (약 1~2분 소요)

✅ BBPE 토크나이저 저장 완료: /gdrive/My Drive/nlpbook/bbpe
   📄 merges.txt  (153,014 bytes)
   📄 vocab.json  (212,853 bytes)
   📄 현재_나의조_최종  (4,096 bytes)


##2. 데모 문장 설정 및 전체 비교 출력
####비교할 문장을 직접 입력합니다.
#####추천: 영화 데이터에서 자주 쓰이는 단어와 신조어·전문용어가 섞인 문장을 다양하게 입력해서 두 방식의 차이를 비교해보세요!

In [ ]:
import pandas as pd

# ── 데모 문장 ────────────────────────────────────────
# ↓ 입력 문장을 바꿔가며 실행해보세요!
DEMO_SENTENCES = [
    "오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음."
]

# ── 비교 시작 ────────────────────────────────────────
for DEMO_SENTENCE in DEMO_SENTENCES:
    print("=" * 80)
    print(f"입력 문장: {DEMO_SENTENCE}")
    print("=" * 80)

    # WordPiece
    wp_tokens = tokenizer_bert.tokenize(DEMO_SENTENCE)
    wp_display = [f"[{t}]" if t.startswith("##") else t for t in wp_tokens]

    # BPE
    bpe_tokens = bbpe_tokenizer.encode(DEMO_SENTENCE).tokens
    bpe_readable = [decode_bbpe_token(t) for t in bpe_tokens]

    print(f"\nBPE ({len(bpe_tokens)}토큰): {bpe_readable}")
    print(f"WP  ({len(wp_tokens)}토큰): {wp_display}")

    # ── DataFrame용 데이터 구성 ───────────────────────
    rows = []
    max_diff = 0
    max_diff_word = ""

    for word in DEMO_SENTENCE.split():
        # BPE
        bpe_w = [decode_bbpe_token(t) for t in bbpe_tokenizer.encode(word).tokens]

        # WordPiece
        wp_w = [f"[{t}]" if t.startswith("##") else t
                for t in tokenizer_bert.tokenize(word)]

        diff = abs(len(bpe_w) - len(wp_w))

        if diff > max_diff:
            max_diff = diff
            max_diff_word = word

        rows.append({
            "어절": word,
            "BPE 토큰": bpe_w,
            "WP 토큰": wp_w,
            "BPE 개수": len(bpe_w),
            "WP 개수": len(wp_w),
            "차이": diff
        })

    # ── DataFrame 출력 ───────────────────────────────
    df = pd.DataFrame(rows)
    display(df)

    if max_diff > 0:
        print(f"\n★ 토큰 수 차이 최대 어절: '{max_diff_word}' (차이: {max_diff})\n")

입력 문장: 오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.

BPE (12토큰): ['오늘', '▁영화', '▁진짜', '▁꿀잼', '이', '었어', '.', '▁연출과', '▁OST', '▁최고', '였음', '.']
WP  (12토큰): ['오늘', '영화', '진짜', '꿀잼', '[##이었]', '[##어]', '.', '연출과', 'OST', '최고', '[##였음]', '.']


,어절,BPE 토큰,WP 토큰,BPE 개수,WP 개수,차이
0,오늘,[오늘],[오늘],1,1,0
1,영화,[영화],[영화],1,1,0
2,진짜,[진짜],[진짜],1,1,0
3,꿀잼이었어.,"[꿀잼, 이, 었어, .]","[꿀잼, [##이었], [##어], .]",4,4,0
4,연출과,"[연출, 과]",[연출과],2,1,1
5,OST,[OST],[OST],1,1,0
6,최고였음.,"[최고, 였음, .]","[최고, [##였음], .]",3,3,0



★ 토큰 수 차이 최대 어절: '연출과' (차이: 1)



##3. BPE vs WordPiece 전체 토큰 비교
####각 문장을 BPE와 WordPiece로 토큰화한 결과를 나란히 출력합니다. 토큰 수와 토큰 내용을 한눈에 비교해보세요.

In [ ]:
import pandas as pd

print("=" * 72)
print(" BPE vs WordPiece 전체 토큰 비교 ")
print("=" * 72)

rows = []

for sent in DEMO_SENTENCES:
    # BPE
    bpe_tokens = bbpe_tokenizer.encode(sent).tokens
    bpe_readable = [decode_bbpe_token(t) for t in bpe_tokens]

    # WordPiece
    wp_tokens = tokenizer_bert.tokenize(sent)
    wp_display = [f"[{t}]" if t.startswith("##") else t for t in wp_tokens]

    rows.append({
        "문장": sent,
        "BPE 토큰": bpe_readable,
        "BPE 개수": len(bpe_readable),
        "WP 토큰": wp_display,
        "WP 개수": len(wp_display)
    })

df = pd.DataFrame(rows)
display(df)

 BPE vs WordPiece 전체 토큰 비교 


,문장,BPE 토큰,BPE 개수,WP 토큰,WP 개수
0,오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.,"[오늘, ▁영화, ▁진짜, ▁꿀잼, 이, 었어, ., ▁연출과, ▁OST, ▁최고, 였음, .]",12,"[오늘, 영화, 진짜, 꿀잼, [##이었], [##어], ., 연출과, OST, 최고, [##였음], .]",12


##4. 어절 경계 표현 비교 (▁ vs ##)
#### BPE는 ▁로 어절의 시작을, WordPiece는 ##으로 이전 토큰과 이어짐을 표시합니다. 같은 문장에서 두 방식이 어절 경계를 어떻게 다르게 표현하는지 확인해보세요.

In [ ]:
import pandas as pd

print("\n" + "=" * 72)
print(" 어절 경계 표현 비교 (▁ vs ##) ")
print("=" * 72)

rows = []

for sent in DEMO_SENTENCES:
    # BPE (수정됨)
    bpe_tokens = bbpe_tokenizer.encode(sent).tokens
    bpe_readable = [decode_bbpe_token(t) for t in bpe_tokens]

    # WordPiece
    wp_tokens = tokenizer_bert.tokenize(sent)
    wp_display = []
    for t in wp_tokens:
        if t.startswith("##"):
            wp_display.append(f"[{t}]")  # 이어지는 토큰
        else:
            wp_display.append(t)        # 새로운 어절

    rows.append({
        "문장": sent,
        "BPE (▁=어절 시작)": bpe_readable,
        "WP (##=이어붙음)": wp_display
    })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', None)
display(df)


 어절 경계 표현 비교 (▁ vs ##) 


,문장,BPE (▁=어절 시작),WP (##=이어붙음)
0,오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.,"[오늘, ▁영화, ▁진짜, ▁꿀잼, 이, 었어, ., ▁연출과, ▁OST, ▁최고, 였음, .]","[오늘, 영화, 진짜, 꿀잼, [##이었], [##어], ., 연출과, OST, 최고, [##였음], .]"


## 5. 어절별 토큰 수 비교
#### 입력 문장의 각 어절을 BPE / WordPiece로 각각 토큰화하여 토큰 수를 나란히 비교합니다. 두 방식 간 차이가 큰 어절을 주목해보세요.

In [ ]:
import pandas as pd

print("\n" + "=" * 72)
print(" [Demo 3] 어절별 토큰 수 비교 ")
print("=" * 72)

all_rows = []

for sent in DEMO_SENTENCES:
    for word in sent.split():
        # BPE (수정됨)
        bpe_w = [decode_bbpe_token(t) for t in bbpe_tokenizer.encode(word).tokens]

        # WordPiece
        wp_w = [f"[{t}]" if t.startswith("##") else t
                for t in tokenizer_bert.tokenize(word)]

        diff = abs(len(bpe_w) - len(wp_w))

        all_rows.append({
            "문장": sent,
            "어절": word,
            "BPE 토큰": bpe_w,
            "WP 토큰": wp_w,
            "BPE 개수": len(bpe_w),
            "WP 개수": len(wp_w),
            "차이": diff
        })

df = pd.DataFrame(all_rows)

# 보기 좋게 설정
pd.set_option('display.max_colwidth', None)

display(df)


 [Demo 3] 어절별 토큰 수 비교 


,문장,어절,BPE 토큰,WP 토큰,BPE 개수,WP 개수,차이
0,오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.,오늘,[오늘],[오늘],1,1,0
1,오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.,영화,[영화],[영화],1,1,0
2,오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.,진짜,[진짜],[진짜],1,1,0
3,오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.,꿀잼이었어.,"[꿀잼, 이, 었어, .]","[꿀잼, [##이었], [##어], .]",4,4,0
4,오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.,연출과,"[연출, 과]",[연출과],2,1,1
5,오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.,OST,[OST],[OST],1,1,0
6,오늘 영화 진짜 꿀잼이었어. 연출과 OST 최고였음.,최고였음.,"[최고, 였음, .]","[최고, [##였음], .]",3,3,0


## 6. 핵심 해석
#### 두 토크나이저의 결과를 종합적으로 정리합니다. 어절 경계 표현 방식과 분할 기준의 차이를 확인해보세요.

In [ ]:
print("\n" + "=" * 72)
print("📌 핵심 해석")
print("=" * 72)

print("- BPE는 ▁ 기호를 사용하여 공백(어절 시작)을 명시적으로 표현한다")
print("- WordPiece는 ## 기호를 사용하여 이전 토큰과 이어지는 subword를 나타낸다")
print("- 동일한 단어라도 두 토크나이저는 서로 다른 방식으로 분해된다")
print("- BPE는 빈도 기반 병합으로 더 세밀하게 분할되는 경향이 있다")
print("- WordPiece는 의미 단위를 유지하면서 분할하려는 경향이 있다")
print("\n" + "=" * 72)
print("- 따라서 BPE는 표현력은 높지만 토큰 수가 증가하고, WordPiece는 효율적인 표현을 지향한다")


📌 핵심 해석
- BPE는 ▁ 기호를 사용하여 공백(어절 시작)을 명시적으로 표현한다
- WordPiece는 ## 기호를 사용하여 이전 토큰과 이어지는 subword를 나타낸다
- 동일한 단어라도 두 토크나이저는 서로 다른 방식으로 분해된다
- BPE는 빈도 기반 병합으로 더 세밀하게 분할되는 경향이 있다
- WordPiece는 의미 단위를 유지하면서 분할하려는 경향이 있다

- 따라서 BPE는 표현력은 높지만 토큰 수가 증가하고, WordPiece는 효율적인 표현을 지향한다


---
## 📝 강의 요약

### 핵심 개념 정리

| 항목 | 내용 |
|------|------|
| **WordPiece** | 우도를 최대화하는 쌍을 병합. 문자 단위 시작이므로 한국어가 그대로 저장됨. |
| **vocab.txt** | 줄 번호 = 인덱스. 특수 토큰 5종이 맨 앞. `##` prefix 서브워드 포함. |
| **`##` prefix** | 어절 내 첫 서브워드 이후의 토큰에 붙는 표시. 어절 경계를 명확히 구분. |
| **`[CLS]`/`[SEP]`** | BERT 입력 시 자동 추가. `[CLS]`는 분류 태스크용, `[SEP]`는 문장 끝/구분. |
| **`[MASK]`** | MLM 학습용. 전체 토큰의 15%를 마스킹하여 BERT가 원래 단어를 예측하도록 훈련. |
| **`[UNK]`** | 어절 전체를 처리할 서브워드가 없을 때 발생. BPE에는 없는 WordPiece 고유 처리. |
| **`token_type_ids`** | 두 문장 입력 시 첫 문장=0, 두 번째 문장=1. NSP 학습에 필수. |
| **BPE vs WordPiece** | 병합 기준, 시작 단위, 저장 방식, 파일 구조, OOV 처리가 모두 다름. |

---

### ❓ 퀴즈로 복습하기

1. WordPiece의 병합 기준인 "우도 최대화"가 BPE의 "빈도 최대화"보다 좋은 점은 무엇인가요?
2. `vocab.txt`에서 `##`으로 시작하는 토큰이 전혀 없다면 어떤 상황인가요?
3. BERT에서 `[MASK]` 토큰을 도입한 이유는 무엇인가요? (GPT의 학습 방식과 비교하세요)
4. WordPiece에서 `[UNK]`가 발생할 때 BPE는 같은 상황을 어떻게 처리하나요?
5. 단일 문장 입력 시 `token_type_ids`가 모두 0인데, 두 문장 입력에서 이 값이 없으면 무슨 문제가 생기나요?

---

### 📚 참고 자료

- [ratsgo's NLPBOOK — Tokenization](https://ratsgo.github.io/nlpbook/docs/preprocess/tokenization/)
- [ratsgo's NLPBOOK — BPE](https://ratsgo.github.io/nlpbook/docs/preprocess/bpe/)
- [ratsgo's NLPBOOK — Vocab Tutorial](https://ratsgo.github.io/nlpbook/docs/preprocess/vocab/)
- [ratsgo's NLPBOOK — Tokenization Tutorial](https://ratsgo.github.io/nlpbook/docs/preprocess/encode/)
- [HuggingFace Tokenizers 공식 문서](https://huggingface.co/docs/tokenizers/)
- [WordPiece 원논문 (Schuster & Nakamura, 2012)](https://static.googleusercontent.com/media/research.google.com/ko//pubs/archive/37842.pdf)
- [BERT 논문 (Devlin et al., 2018)](https://arxiv.org/abs/1810.04805)
